<a href="https://colab.research.google.com/github/YanisaChandang/-Spark-Hack/blob/main/OneWaterAI_KoSichang_WebApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneWaterAI · เกาะสีชัง — B01–B12 GIS revision

Run installation cell in Colab. If this session already imported NumPy before installation, restart the session once, then run the application cell. Open the temporary Gradio URL; keep the Colab session running.

The notebook embeds the original R/D8 12 basin boundaries, runoff paths and modeled coastal outlets. There are no required external GIS files. Geometry is regenerated from the original 30 September 2026 R source, AWS/Mapzen Skadi N13E100 elevation grid (~30 m) and OpenStreetMap island boundary (object 13094563). These are modeled drainage zones/outlets, not surveyed rivers, official delineations or confirmed sampling stations.

All water-quality measurements, scores, forecasts and model metrics are synthetic and unvalidated. Basin colors distinguish IDs; point colors show synthetic scores. The optional risk shading joins synthetic scores to basins by B-ID only; it is not evidence of uniform contamination within a basin or offshore risk extent.

The table palette now stays light and legible in both light and dark browser modes. Optional GIS uploads override supplied layers only. Terrain and groundwater rasters remain optional and are not supplied by this revision.


In [ ]:
# Compatible with current Google Colab Python 3.13.
# NumPy 2.2.6 fails while importing numpy.strings on this runtime.
%pip -q install --upgrade --force-reinstall gradio==5.49.1 numpy==2.3.3 pandas==2.3.3 scipy==1.16.2 scikit-learn==1.7.2 plotly==6.3.0 rasterio==1.4.3 pillow==11.3.0 shapely==2.1.1 huggingface-hub==0.35.3

# REQUIRED after this cell finishes: Runtime > Restart session.
# Then run only the complete application cell below.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.

In [ ]:
import csv
import uuid
import shutil
import threading
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning


# ─────────────────────── Configuration ───────────────────────

SEED = 20261016
IDS = [f"B{i:02d}" for i in range(1, 13)]
DEFAULT = "B05"
OUTLET_COORDS = {'B01': [100.806666666667, 13.1422222222222], 'B02': [100.809722222222, 13.1605555555556], 'B03': [100.816666666667, 13.1338888888889], 'B04': [100.808611111111, 13.1619444444444], 'B05': [100.817777777778, 13.1425], 'B06': [100.807777777778, 13.1652777777778], 'B07': [100.811111111111, 13.1272222222222], 'B08': [100.8125, 13.1566666666667], 'B09': [100.814722222222, 13.1525], 'B10': [100.806944444444, 13.1458333333333], 'B11': [100.817222222222, 13.15], 'B12': [100.817777777778, 13.1483333333333]}

HUMAN = ["HF183", "crAssphage", "PMMoV"]
ANIMAL = ["C16S", "qC160F"]
BIO = ["MiFish", "MollCOI253", "Ceph16S", "Crust16S"]
PATHOGEN = ["Norovirus GI", "Norovirus GII", "VP28", "YHV-1", "AHPND", "NNV"]

MICROBES = HUMAN + ANIMAL + BIO + PATHOGEN
ENV = ["Rainfall", "Temperature", "DO", "pH"]
EDIT = ENV + MICROBES
LAGS = ["rain_lag1", "rain_lag3", "rain_prev7", "dry_before", "first_rain"]
NUM = EDIT + LAGS
XCOLS = ["site_id"] + NUM

GROUPS = [
    ("ตัวบ่งชี้ที่เกี่ยวข้องกับอุจจาระมนุษย์", HUMAN),
    ("ตัวบ่งชี้สัตว์บก/สัตว์ปีก — candidate assays", ANIMAL),
    ("ความหลากหลายทางชีวภาพ / eDNA", BIO),
    ("เป้าหมายเชื้อก่อโรค — แยกจาก MST", PATHOGEN),
]

RANGES = {
    "Rainfall": (0, 120),
    "Temperature": (22, 36),
    "DO": (0.5, 12),
    "pH": (6, 9),
}
ENV_LABELS = [
    "Rainfall — ฝน (mm)",
    "Temperature — อุณหภูมิน้ำ (°C)",
    "DO — ออกซิเจนละลาย (mg/L)",
    "pH",
]

CATS = ["ต่ำ · สาธิต", "ปานกลาง · สาธิต", "สูง · สาธิต"]
COLORS = dict(zip(CATS, ["#16877e", "#c29539", "#bc5651"]))
SCENARIOS = ["บล็อกข้อมูลย้อนหลัง", "ทดสอบฝนแรกหลังแล้ง 7 วัน"]

OUT = Path.cwd() / "onewaterai_outputs"
OUT.mkdir(exist_ok=True)
SAVE_LOCK = threading.Lock()


def category(score):
    return CATS[0] if score < 35 else CATS[1] if score < 65 else CATS[2]


def action(score):
    if score >= 65:
        return "ส่งผู้รับผิดชอบทบทวนข้อมูลและวางแผนตรวจยืนยัน"
    if score >= 35:
        return "ทบทวนข้อมูลที่ยังขาดและกำหนดการติดตาม"
    return "บันทึกข้อมูลและติดตามตามแผน — ยังไม่ยืนยันความปลอดภัย"


# ─────────────────────── Synthetic daily data ───────────────────────

def features(frame):
    """สร้าง lag จากฝนก่อนหน้าในจุดเดียวกันเท่านั้น"""
    d = frame.copy().sort_values(["site_id", "date"]).reset_index(drop=True)
    g = d.groupby("site_id")["Rainfall"]

    d["rain_lag1"] = g.shift(1).fillna(0)
    d["rain_lag3"] = g.shift(3).fillna(0)
    d["rain_prev7"] = g.transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum()
    ).fillna(0)

    d["dry_before"] = 0
    for _, part in d.groupby("site_id"):
        count, counts = 0, []
        for rain in part["Rainfall"]:
            counts.append(count)
            count = count + 1 if rain < 1 else 0
        d.loc[part.index, "dry_before"] = counts

    d["first_rain"] = (
        (d["Rainfall"] >= 10) & (d["dry_before"] >= 5)
    ).astype(int)
    return d


def generate_data():
    rng = np.random.default_rng(SEED)
    dates = pd.date_range(end="2026-10-15", periods=450)
    season = np.sin(2 * np.pi * np.arange(len(dates)) / 365)

    regional_rain = np.where(
        rng.random(len(dates)) < 0.28 + 0.12 * season,
        rng.gamma(1.7, 13, len(dates)),
        0,
    )

    frames = []
    for i, site in enumerate(IDS):
        d = pd.DataFrame({"date": dates, "site_id": site})
        d["site_name"] = f"จุดระบายของลุ่มรับน้ำ {IDS[i]}"

        # จุดระบายคำนวณจาก D8 เดิม ไม่ใช่สถานีตรวจวัดที่สำรวจยืนยัน
        d["latitude"] = OUTLET_COORDS[IDS[i]][1]
        d["longitude"] = OUTLET_COORDS[IDS[i]][0]

        human_base = 10 + 75 * i / 11
        animal_base = rng.uniform(8, 75)
        aquatic_base = rng.uniform(8, 85)

        d["Rainfall"] = np.clip(
            regional_rain * rng.uniform(0.8, 1.2), 0, 120
        )
        d["Temperature"] = np.clip(
            29 + 2 * season + rng.normal(0, 0.7, len(d)), 24, 34
        )
        d["DO"] = np.clip(
            8.8 - 0.05 * human_base - 0.02 * aquatic_base
            - 0.12 * (d.Temperature - 26)
            + 0.012 * d.Rainfall
            + rng.normal(0, 0.5, len(d)),
            1, 10,
        )
        d["pH"] = np.clip(
            8 + 0.15 * season - 0.003 * d.Rainfall
            + rng.normal(0, 0.13, len(d)),
            6.6, 8.8,
        )

        d = features(d)
        delayed = (
            0.6 * np.tanh(d.rain_lag1 / 20)
            + 0.4 * np.tanh(d.rain_lag3 / 20)
        )
        human = (
            human_base + 14 * delayed
            + 18 * d.first_rain * np.tanh(d.Rainfall / 20)
            - 15 * np.tanh(d.Rainfall / 40)
        )
        animal = animal_base + 9 * delayed - 9 * np.tanh(d.Rainfall / 40)
        aquatic = aquatic_base + 9 * season + 4 * np.maximum(0, 5 - d.DO)

        for name in HUMAN:
            d[name] = np.clip(human + rng.normal(0, 8, len(d)), 0, 100)
        for name in ANIMAL:
            d[name] = np.clip(animal + rng.normal(0, 9, len(d)), 0, 100)
        for name in BIO:
            d[name] = np.clip(
                65 - 0.2 * aquatic + 10 * season
                + rng.normal(0, 12, len(d)), 0, 100
            )
        for name in PATHOGEN:
            base = human if name.startswith("Norovirus") else aquatic
            d[name] = np.clip(base + rng.normal(0, 10, len(d)), 0, 100)

        # สูตรสาธิตที่แต่งขึ้น ไม่ใช่สมการความเสี่ยงที่ผ่าน validation
        h = d[HUMAN].mean(axis=1) / 100
        a = d[ANIMAL].mean(axis=1) / 100
        ph = d[PATHOGEN[:2]].mean(axis=1) / 100
        pa = d[PATHOGEN[2:]].mean(axis=1) / 100
        stress = np.clip((6 - d.DO) / 5, 0, 1)

        # ฝนมีทั้งพจน์บวกและลบ ขึ้นกับบริบทจำลอง
        rain_effect = (
            0.08 * d.first_rain * np.tanh(d.Rainfall / 30)
            + 0.10 * np.tanh(d.rain_prev7 / 60) * (1 - d.DO / 10)
            - 0.10 * np.tanh(d.Rainfall / 40) * (d.DO / 10)
        )
        score = 100 * (
            0.03 + 0.20*h + 0.08*a + 0.26*ph + 0.15*pa
            + 0.16*stress + 0.04*abs(d.pH - 8)/1.5
            + 0.02*np.maximum(d.Temperature - 30, 0)/5
            + rain_effect
        )

        # Biodiversity/eDNA ไม่มีน้ำหนักเพิ่มคะแนนโดยตรง
        d["risk_score"] = np.clip(
            score + rng.normal(0, 2.2, len(d)), 0, 100
        )
        d["data_status"] = "SYNTHETIC_ONLY"
        frames.append(d)

    return pd.concat(frames, ignore_index=True)


DATA = generate_data()
LATEST = (
    DATA.sort_values("date").groupby("site_id").tail(1)
    .set_index("site_id").loc[IDS]
)
LATEST["risk_category"] = LATEST.risk_score.map(category)
LATEST["recommended_action"] = LATEST.risk_score.map(action)
DATA.to_csv(OUT / "synthetic_daily.csv", index=False, encoding="utf-8-sig")


# ─────────────────────── Train comparable models ───────────────────────

# ตัดช่วงเริ่มต้นที่ยังมีประวัติ lag ไม่ครบ
eligible = DATA[DATA.groupby("site_id").cumcount() >= 7].copy()
dates = np.sort(eligible.date.unique())
CUTOFF = pd.Timestamp(dates[int(0.70 * len(dates))])

# ทุกจุดใช้เส้นแบ่งวันที่เดียวกัน ป้องกันการสุ่มวันอนาคตเข้า train
TRAIN = eligible[eligible.date < CUTOFF]
TEST = eligible[eligible.date >= CUTOFF]
TRAIN_MEANS = TRAIN[NUM].mean()

MODELS, evaluation = {}, []
estimators = [
    ("Simple Regression (Ridge)", Ridge(alpha=5)),
    (
        "Neural Network (MLPRegressor)",
        MLPRegressor(
            hidden_layer_sizes=(32, 16),
            max_iter=400,
            early_stopping=False,
            n_iter_no_change=25,
            random_state=SEED,
        ),
    ),
]

for name, estimator in estimators:
    preprocess = ColumnTransformer([
        ("numeric", StandardScaler(), NUM),
        (
            "site",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["site_id"],
        ),
    ])
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocess", preprocess),
            ("model", estimator),
        ]),
        transformer=StandardScaler(),
    )

    with warnings.catch_warnings(record=True) as records:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(TRAIN[XCOLS], TRAIN.risk_score)

    predicted = np.clip(model.predict(TEST[XCOLS]), 0, 100)
    status = (
        "ถึง max_iter; ควรทบทวน convergence"
        if any(issubclass(w.category, ConvergenceWarning) for w in records)
        else "จบการฝึก"
    )
    evaluation.append([
        name,
        r2_score(TEST.risk_score, predicted),
        mean_absolute_error(TEST.risk_score, predicted),
        np.sqrt(mean_squared_error(TEST.risk_score, predicted)),
        status,
    ])
    MODELS[name] = model

MODEL_NAMES = list(MODELS)
METRICS = pd.DataFrame(
    evaluation, columns=["โมเดล", "R²", "MAE", "RMSE", "สถานะ"]
)


def predict(model_name, rows):
    return np.clip(MODELS[model_name].predict(rows[XCOLS]), 0, 100)


# ─────────────────────── Dashboard helpers ───────────────────────

def styled(fig, title, height=380):
    fig.update_layout(title=dict(text=title,font=dict(size=14,color='#193b48')),
        template='plotly_white',height=height,
        font=dict(family='Noto Sans Thai, Tahoma, Arial',size=11,color='#607f8c'),
        paper_bgcolor='white',plot_bgcolor='white',
        margin=dict(l=35,r=25,t=50,b=40),
        legend=dict(orientation='h',y=-.18,x=0),
        modebar_remove=['lasso2d','select2d','autoScale2d'],hoverlabel=dict(bgcolor='white'))
    fig.update_xaxes(gridcolor='#edf2f2',zerolinecolor='#e1eaea')
    fig.update_yaxes(gridcolor='#edf2f2',zerolinecolor='#e1eaea')
    return fig


def indicator_table(row, names, environmental=False):
    units = dict(zip(ENV, ["mm", "°C", "mg/L", "ไม่มีหน่วย"]))
    return pd.DataFrame({
        "Indicator": names,
        "ค่า": [round(float(row[n]), 2) for n in names],
        "หน่วย": [
            units[n] if environmental else "ดัชนีจำลอง 0–100"
            for n in names
        ],
    })


def home_view(site):
    row = LATEST.loc[site]
    hist = DATA[DATA.site_id == site]
    text = (
        f"### {site} — {row.site_name}\n"
        f"**คะแนน {row.risk_score:.1f}/100 · {category(row.risk_score)}**  \n"
        f"{action(row.risk_score)}  \n"
        f"วันที่ข้อมูลจำลอง: {row.date:%d/%m/%Y}"
    )
    fig = px.line(
        hist, x="date", y="risk_score",
        labels={"date": "วันที่", "risk_score": "คะแนนสาธิต"},
    )
    fig.update_traces(line_color="#16877e")
    fig.update_yaxes(range=[0, 100])

    return (
        text,
        *[indicator_table(row, cols) for _, cols in GROUPS],
        indicator_table(row, ENV, True),
        styled(fig, f"แนวโน้มรายวัน — {site}"),
    )


# ─────────────────────── Conditional 30-day scenarios ───────────────────────

def make_future(site, scenario):
    hist = DATA[DATA.site_id == site].sort_values("date")
    future_dates = pd.date_range(
        hist.date.max() + pd.Timedelta(days=1), periods=30
    )

    # สุ่มบล็อก 30 วันที่ทราบแล้ว โดยพยายามจับคู่เดือนตามปฏิทิน
    starts = [
        i for i in range(len(hist) - 29)
        if hist.iloc[i].date.month == future_dates[0].month
    ]
    rng = np.random.default_rng(SEED + IDS.index(site))
    start = int(rng.choice(starts if starts else np.arange(len(hist) - 29)))
    source = hist.iloc[start:start + 30]

    # ใช้เฉพาะ predictors ไม่มี risk_score จากบล็อกเดิมส่งเข้าโมเดล
    future = source[["site_id"] + EDIT].copy().reset_index(drop=True)
    future["date"] = future_dates

    if scenario == SCENARIOS[1]:
        future.loc[:6, "Rainfall"] = 0
        future.loc[7, "Rainfall"] = 30

    combined = pd.concat([
        hist[["date", "site_id"] + EDIT], future
    ], ignore_index=True)

    return features(combined).tail(30).copy(), source


def forecast_view(site, model_name, scenario):
    hist = DATA[DATA.site_id == site].sort_values("date")
    future, source = make_future(site, scenario)
    future["prediction"] = predict(model_name, future)
    recent = hist.tail(90)

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    rain = pd.concat([
        recent[["date", "Rainfall"]],
        future[["date", "Rainfall"]],
    ])
    fig.add_trace(
        go.Bar(
            x=rain.date, y=rain.Rainfall,
            name="Rainfall (mm)", marker_color="#91c7d6", opacity=0.30,
        ),
        secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=recent.date, y=recent.risk_score,
            name="คะแนนย้อนหลังจำลอง", line=dict(color="#16877e"),
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=future.date, y=future.prediction,
            name="คาดการณ์ตามสมมติฐาน",
            line=dict(color="#bc5651", dash="dash"),
        ),
        secondary_y=False,
    )

    events = future[future.first_rain == 1]
    fig.add_trace(
        go.Scatter(
            x=events.date, y=events.prediction, mode="markers",
            marker=dict(color="#c29539", size=10, symbol="diamond"),
            name="ฝนแรกหลังแล้ง ≥5 วัน",
        ),
        secondary_y=False,
    )
    fig.add_vline(
        x=hist.date.max().timestamp() * 1000,
        line_dash="dot", line_color="#64748b",
    )
    fig.update_yaxes(
        title_text="คะแนนสาธิต", range=[0, 100], secondary_y=False
    )
    fig.update_yaxes(
        title_text="Rainfall (mm)", rangemode="tozero", secondary_y=True
    )

    table = pd.DataFrame({
        "วันที่": future.date.dt.strftime("%Y-%m-%d"),
        "ฝนสมมติ (mm)": future.Rainfall.round(1),
        "ฝน lag 1 วัน": future.rain_lag1.round(1),
        "ฝน lag 3 วัน": future.rain_lag3.round(1),
        "ฝนสะสม 7 วันก่อนหน้า": future.rain_prev7.round(1),
        "วันแห้งก่อนหน้า": future.dry_before.astype(int),
        "ฝนแรกหลังแล้ง": future.first_rain.map({0: "ไม่ใช่", 1: "ใช่"}),
        "คะแนนคาดการณ์": future.prediction.round(1),
        "ระดับ": future.prediction.map(category),
    })

    assumption = (
        "กรณีทดสอบนี้ตั้งฝน 7 วันแรกเป็น 0 และวันที่ 8 เป็น 30 mm; "
        "ตัวแปรอื่นคงตามบล็อกสมมติ"
        if scenario == SCENARIOS[1]
        else "กรณีนี้ใช้ค่าฝนและตัวแปรอื่นตามบล็อกสมมติ "
             "โดยไม่บังคับเหตุ first-rain"
    )
    note = (
        f"**สถานการณ์สมมติ ไม่ใช่พยากรณ์อากาศ:** "
        f"ใช้บล็อกย้อนหลังของจุดเดียวกัน "
        f"({source.date.min():%d/%m/%Y}–{source.date.max():%d/%m/%Y}) "
        f"เป็นค่าอนาคต 30 วัน รวม environmental/MST/eDNA/pathogen inputs "
        f"ไม่ได้ใช้ผลตรวจในอนาคต  \n"
        f"{assumption} จึงเป็น conditional scenario "
        f"ไม่ใช่แบบจำลองการแพร่เชื้อ  \n"
        f"lag ใช้เฉพาะฝนก่อนวันนั้น · ช่วงอนาคตพบ first-rain "
        f"{len(events)} วัน โมเดลเรียนรู้ความสัมพันธ์จากข้อมูล "
        f"ไม่กำหนดให้ฝนเพิ่มความเสี่ยงเสมอ"
    )
    return (
        styled(fig, f"{site} · {model_name} · 30 วัน", 450),
        table,
        note,
    )


# ─────────────────────── Rule-based community tasks ───────────────────────

ROLE_TASKS = {
    "ผู้บริหารนโยบาย": [
        "กำหนดเจ้าของงานและตรวจช่องว่างข้อมูล",
        "ทบทวนทรัพยากรสำหรับการตรวจยืนยัน",
        "ประสานหน่วยงานตามขอบเขตหน้าที่",
        "กำหนดวันติดตามและหลักฐานที่ต้องทบทวน",
    ],
    "นักท่องเที่ยว": [
        "ตรวจประกาศพื้นที่ก่อนทำกิจกรรม",
        "บันทึกสิ่งผิดปกติจากพื้นที่ที่เข้าถึงได้อย่างปลอดภัย",
        "ส่งข้อมูลให้ผู้ดูแลพื้นที่โดยไม่ระบุข้อมูลส่วนบุคคล",
        "ติดตามประกาศและคำแนะนำของพื้นที่",
    ],
    "เจ้าหน้าที่ดูแลแหล่งน้ำ": [
        "ตรวจคุณภาพข้อมูลและสถานะเครื่องมือ",
        "เสนอแผนตรวจยืนยันตาม SOP ที่อนุมัติ",
        "ประสานการจัดการตามแผนของหน่วยงาน",
        "บันทึกผลติดตามก่อนเสนอเปลี่ยนมาตรการ",
    ],
    "ประชาชนในพื้นที่": [
        "บันทึกตำแหน่งและเวลาที่พบปัญหา",
        "ระบุกิจกรรมใช้น้ำและการเปลี่ยนแปลงที่สังเกต",
        "ส่งรายงานให้ผู้ประสานงานพื้นที่",
        "ติดตามผลยืนยันและประกาศของพื้นที่",
    ],
    "ผู้นำชุมชน": [
        "รวมรายงานที่เกี่ยวข้องกับเหตุเดียวกัน",
        "กำหนดผู้ประสานงานและข้อมูลที่ยังขาด",
        "นัดหมายการติดตามร่วมกับผู้รับผิดชอบ",
        "สื่อสารสถานะโดยแยกสิ่งที่ทราบกับสิ่งที่ยังไม่ยืนยัน",
    ],
    "เกษตรกรผู้เลี้ยงปลา": [
        "ตรวจระบบน้ำตาม SOP ของฟาร์มที่เกี่ยวข้อง",
        "รวบรวมประวัติระบบน้ำและสุขภาพสัตว์น้ำ",
        "ส่งข้อมูลให้ผู้เชี่ยวชาญประเมินโดยไม่เลือกยาเอง",
        "บันทึกการดำเนินงานและผลติดตามตามแผน",
    ],
    "ชาวประมง": [
        "ตรวจประกาศที่เกี่ยวข้องกับกิจกรรมประมง",
        "บันทึกความผิดปกติระหว่างกิจกรรมปกติ",
        "ส่งข้อมูลพื้นที่ให้ผู้รับผิดชอบทบทวน",
        "ติดตามผลตรวจ โดยไม่ใช้คะแนนนี้เลือกทะเลที่ไกลหรือลึกกว่า",
    ],
}
ROLES = list(ROLE_TASKS)
ACTION_NOTE = (
    "ข้อเสนอแนะเหล่านี้เป็นแนวทางเบื้องต้น\n"
    "ซึ่งสามารถพัฒนาร่วมกับชุมชนเพื่อให้เหมาะสม\n"
    "กับบริบทพื้นที่และระดับความเสี่ยงในอนาคต"
)


def task_context(site, role):
    row = LATEST.loc[site]
    relevant = (
        f"Rainfall {row.Rainfall:.1f} mm; "
        f"DO {row.DO:.1f} mg/L; pH {row.pH:.2f}"
    )
    flags = []
    if row.DO < 4:
        flags.append("ทบทวนค่า DO และระบบตรวจวัดตาม SOP")
    if row.Rainfall >= 20 or row.first_rain:
        flags.append("ตรวจข้อมูลฝนก่อนหน้าและผลหน่วงเวลา")
    if not 7 <= row.pH <= 8.5:
        flags.append("ตรวจค่า pH ซ้ำและที่มาของข้อมูล")
    if row.Temperature > 32:
        flags.append("ทบทวนข้อมูลอุณหภูมิและบริบทการใช้น้ำ")

    tasks = list(ROLE_TASKS[role])
    tasks[0] += f" — {category(row.risk_score)}: {action(row.risk_score)}"
    tasks[1] += (
        " — "
        + ("; ".join(flags) if flags else "ติดตามตัวแปรตามแผนพื้นที่")
        + f" ({relevant}; เกณฑ์คัดงานสมมติ)"
    )
    return dict(
        site=site, role=role, score=float(row.risk_score),
        category=category(row.risk_score), tasks=tasks,
    )


def task_headers(ctx):
    return [
        f"### ภารกิจที่ต้องทำอันดับที่ {i+1}\n{task}"
        for i, task in enumerate(ctx["tasks"])
    ]


def task_summary(ctx):
    return (
        f"**{ctx['site']} · {ctx['role']} · "
        f"คะแนนปัจจุบันจำลอง {ctx['score']:.1f}/100 · {ctx['category']}**\n"
        "งานอ้างอิงคะแนนปัจจุบันจากสูตรสาธิต ไม่ใช่ผลคาดการณ์อนาคต"
    )


def reset_tasks(site, role):
    ctx = task_context(site, role)
    return (
        ctx, task_summary(ctx), *task_headers(ctx),
        *([""] * 4), *([None] * 4), "", None,
    )


def save_tasks(ctx, site, role, *payload):
    if not ctx or ctx["site"] != site or ctx["role"] != role:
        raise gr.Error(
            "พื้นที่หรือบทบาทเพิ่งเปลี่ยน โปรดรอให้ภารกิจอัปเดตก่อนส่ง"
        )

    texts, images = payload[:4], payload[4:]
    if any(
        not isinstance(t, str) or not t.strip() or len(t) > 2000
        for t in texts
    ):
        raise gr.Error(
            "กรอกคำตอบทั้ง 4 ข้อ ข้อละไม่เกิน 2,000 ตัวอักษร; "
            "รูปเป็นตัวเลือก"
        )

    submission = uuid.uuid4().hex
    folder = OUT / submission
    folder.mkdir()
    rows = []

    for i, (text, image) in enumerate(zip(texts, images)):
        image_name = ""
        if image:
            source = Path(image)
            if not source.is_file() or source.stat().st_size > 10 * 1024 * 1024:
                raise gr.Error(
                    "รูปต้องเป็นไฟล์ที่อัปโหลดและมีขนาดไม่เกิน 10 MB"
                )
            destination = folder / f"task_{i+1}{source.suffix.lower()}"
            shutil.copy2(source, destination)
            image_name = str(destination.relative_to(OUT))

        rows.append(dict(
            submission_id=submission,
            timestamp_utc=datetime.now(timezone.utc).isoformat(),
            site_id=site,
            role=role,
            current_demo_risk=ctx["score"],
            risk_category=ctx["category"],
            task_number=i+1,
            task=ctx["tasks"][i],
            response=text.strip(),
            image=image_name,
            status="submitted_unreviewed",
            data_status="SYNTHETIC_DEMO",
        ))

    # ป้องกันข้อความกลายเป็นสูตรเมื่อเปิด CSV ใน spreadsheet
    for row in rows:
        for key, value in row.items():
            if isinstance(value, str) and value.lstrip().startswith(
                ("=", "+", "-", "@")
            ):
                row[key] = "'" + value

    receipt = folder / "receipt.csv"
    with SAVE_LOCK:
        pd.DataFrame(rows).to_csv(
            receipt, index=False, encoding="utf-8-sig"
        )
        aggregate = OUT / "submissions.csv"
        with aggregate.open("a", newline="", encoding="utf-8-sig") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            if stream.tell() == 0:
                writer.writeheader()
            writer.writerows(rows)

    return (
        f"✅ บันทึก 4 ภารกิจแล้ว · รหัส {submission[:10]} · "
        "รอตรวจทาน ยังไม่ยืนยันคุณภาพน้ำ",
        str(receipt),
    )


# ─────────────────────── Trained-model simulator ───────────────────────

def simulate(site, model_name, *values):
    if site not in IDS or model_name not in MODELS or len(values) != len(EDIT):
        raise gr.Error("โปรดเลือกจุด โมเดล และกรอก input ให้ครบ")

    original = LATEST.loc[site]
    row = original.copy()

    for name, value in zip(EDIT, values):
        lo, hi = RANGES.get(name, (0, 100))
        if value is None or not np.isfinite(value) or not lo <= float(value) <= hi:
            raise gr.Error(
                f"{name}: โปรดระบุตัวเลขระหว่าง {lo}–{hi}"
            )
        row[name] = float(value)

    row["site_id"] = site
    row["first_rain"] = int(
        row.Rainfall >= 10 and row.dry_before >= 5
    )
    base_row = original.copy()
    base_row["site_id"] = site

    new = float(predict(model_name, pd.DataFrame([row]))[0])
    baseline = float(predict(model_name, pd.DataFrame([base_row]))[0])

    # Sensitivity: แทนทีละตัวด้วยค่าเฉลี่ย TRAIN
    alternatives = []
    for name in EDIT:
        alternate = row.copy()
        alternate[name] = float(TRAIN_MEANS[name])
        alternate["first_rain"] = int(
            alternate.Rainfall >= 10 and alternate.dry_before >= 5
        )
        alternatives.append(alternate)

    deltas = new - predict(model_name, pd.DataFrame(alternatives))
    ranking = pd.DataFrame({"ตัวแปร": EDIT, "ผลต่างคะแนน": deltas})
    ranking = ranking.iloc[
        np.argsort(abs(ranking["ผลต่างคะแนน"].to_numpy()))[-8:]
    ]

    fig = go.Figure(go.Bar(
        x=ranking["ผลต่างคะแนน"],
        y=ranking["ตัวแปร"],
        orientation="h",
        marker_color=[
            "#bc5651" if v >= 0 else "#16877e"
            for v in ranking["ผลต่างคะแนน"]
        ],
    ))
    fig.add_vline(x=0, line_color="#64748b")
    fig.update_xaxes(
        title="Δ score vs training-mean replacement"
    )

    note = (
        f"ค่าทำนายของโมเดลเมื่อไม่แก้ input = **{baseline:.1f}**; "
        f"ผลต่างจากการแก้ input ภายในโมเดลเดียวกัน = "
        f"**{new-baseline:+.1f}** คะแนน  \n"
        "ผลต่างเทียบคะแนนอ้างอิงด้านบนรวมความคลาดเคลื่อนของโมเดลด้วย  \n"
        f"**Action สาธิต:** {action(new)}  \n"
        "**ไม่ใช่เหตุและผล:** sensitivity เปลี่ยนทีละตัวไปค่าเฉลี่ย train; "
        "ตัวแปรที่สัมพันธ์กันอาจให้สถานการณ์ที่ไม่เกิดจริง "
        "ส่วนต่างไม่จำเป็นต้องรวมเท่าคะแนน · "
        "lag ฝนก่อนหน้าคงตามประวัติของจุดนั้น"
    )
    return (
        float(original.risk_score),
        new,
        new - float(original.risk_score),
        category(new),
        styled(fig, "อะไรมีผลต่อคะแนนครั้งนี้?"),
        note,
    )


def load_sim_site(site):
    row = LATEST.loc[site]
    lag_note = (
        f"ฝน lag 1 วัน {row.rain_lag1:.1f} mm · "
        f"lag 3 วัน {row.rain_lag3:.1f} mm · "
        f"ฝนสะสม 7 วันก่อนหน้า {row.rain_prev7:.1f} mm · "
        f"วันแห้งก่อนหน้า {int(row.dry_before)} วัน"
    )
    return (*[float(row[k]) for k in EDIT], lag_note)


# ─────────────────────── Logical consistency checks ───────────────────────

assert TRAIN.date.max() < TEST.date.min()
assert "risk_score" not in XCOLS

sample = DATA[DATA.site_id == IDS[0]].sort_values("date")
assert np.allclose(
    sample.rain_lag1.iloc[1:], sample.Rainfall.iloc[:-1]
)

for site in IDS:
    future_check, _ = make_future(site, SCENARIOS[1])
    assert len(future_check) == 30
    assert int(future_check.iloc[7].first_rain) == 1

for name in MODEL_NAMES:
    checked = predict(name, TEST.head(12))
    assert np.isfinite(checked).all()
    assert ((checked >= 0) & (checked <= 100)).all()




# Optional GIS ingestion. Uploaded layers affect maps only, never ML inputs/targets.
import json
import html
import base64
import io
import math
import os

GIS_DIR = Path('/content/gis') if Path('/content').exists() else Path.cwd() / 'gis'
GIS_FILES = {'catchments': 'catchments_B01_B12.geojson', 'flows': 'flow_paths.geojson',
             'stations': 'monitoring_sites.geojson', 'slope': 'terrain_slope.tif',
             'groundwater': 'groundwater_screening.tif'}
LAYER_NAMES = ['Drainage catchments', 'Flow paths', 'Monitoring points']
RASTER_NAMES = ['None', 'Terrain slope', 'Groundwater screening — UNVALIDATED']


def checked_path(path, limit=100):
    p = Path(path)
    if not p.is_file() or p.stat().st_size > limit * 1024**2:
        raise ValueError(f'File missing or larger than {limit} MB')
    return p


def validate_position(x, y):
    # Deliberately local to Ko Sichang; rejects most projected/unrelated coordinates.
    if not (math.isfinite(x) and math.isfinite(y) and 100.65 <= x <= 101.0 and 13.0 <= y <= 13.30):
        raise ValueError('Expected WGS84 lon,lat near Ko Sichang (100.65–101.0 E, 13.0–13.30 N). Reproject in R/GIS first.')


def coordinates_iter(coords):
    if not isinstance(coords, (list, tuple)) or not coords:
        raise ValueError('Empty or malformed geometry')
    if isinstance(coords[0], (int, float)):
        if len(coords) < 2:
            raise ValueError('A coordinate needs longitude and latitude')
        validate_position(float(coords[0]), float(coords[1]))
        yield coords[:2]
    else:
        for child in coords:
            yield from coordinates_iter(child)


def read_geojson(path, kind):
    from shapely.geometry import shape
    obj = json.loads(checked_path(path, 20).read_text(encoding='utf-8-sig'))
    crs = obj.get('crs')
    if crs and not any(s in str(crs).upper() for s in ['4326', 'CRS84']):
        raise ValueError('Export GeoJSON in EPSG:4326, longitude then latitude')
    feats = obj.get('features', [])
    if obj.get('type') != 'FeatureCollection' or not 1 <= len(feats) <= 20000:
        raise ValueError('Expected a non-empty GeoJSON FeatureCollection, at most 20,000 features')
    allowed = {'catchments': ['Polygon', 'MultiPolygon'], 'flows': ['LineString', 'MultiLineString'], 'stations': ['Point']}[kind]
    ids, ncoords = [], 0
    for f in feats:
        g = f.get('geometry') or {}
        if g.get('type') not in allowed:
            raise ValueError(f'{kind} requires {allowed}')
        ncoords += sum(1 for _ in coordinates_iter(g.get('coordinates')))
        if ncoords > 300000:
            raise ValueError('Simplify this layer to at most 300,000 coordinates before uploading')
        geom = shape(g)
        if geom.is_empty or not geom.is_valid:
            raise ValueError('Invalid geometry; repair in GIS software before export')
        if kind != 'flows':
            sid = f.get('properties', {}).get('site_id')
            if sid not in IDS:
                raise ValueError('Each catchment/station requires properties.site_id B01–B12')
            ids.append(sid)
        # Keep only required attributes. Arbitrary uploaded HTML is not rendered.
        f['properties'] = {'site_id': f.get('properties', {}).get('site_id', '')}
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate site_id; dissolve catchments to one (Multi)Polygon per B ID')
    if kind == 'catchments' and set(ids) != set(IDS):
        raise ValueError('Catchments must include exactly B01–B12')
    return {'type': 'FeatureCollection', 'features': feats}


def raster_layer(path, kind):
    import rasterio
    from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
    from PIL import Image
    with rasterio.open(checked_path(path)) as src:
        if src.crs is None or src.count != 1:
            raise ValueError('GeoTIFF needs a defined CRS and exactly one numeric band')
        if src.width * src.height > 100_000_000:
            raise ValueError('Raster too large; crop/resample to Ko Sichang first')
        west, south, east, north = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
        validate_position(west, south)
        validate_position(east, north)
        if east <= west or north <= south:
            raise ValueError('Invalid raster extent')
        # Reproject to Web Mercator so the image overlay matches the tile map.
        tr, width, height = calculate_default_transform(src.crs, 'EPSG:3857', src.width, src.height, *src.bounds)
        factor = min(1, 600 / max(width, height))
        out_w, out_h = max(1, round(width * factor)), max(1, round(height * factor))
        tr = tr * rasterio.Affine.scale(width / out_w, height / out_h)
        dst = np.full((out_h, out_w), np.nan, dtype=np.float32)
        reproject(source=rasterio.band(src, 1), destination=dst,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=tr, dst_crs='EPSG:3857', dst_nodata=np.nan,
                  resampling=Resampling.nearest)
        left, bottom, right, top = rasterio.transform.array_bounds(out_h, out_w, tr)
        west, south, east, north = transform_bounds('EPSG:3857', 'EPSG:4326', left, bottom, right, top)
    valid = np.isfinite(dst)
    if not valid.any():
        raise ValueError('Raster has no valid cells')
    if kind == 'slope':
        if np.any((dst[valid] < 0) | (dst[valid] > 90)):
            raise ValueError('Slope raster must be degrees, 0–90; set NoData correctly')
        # Fixed 0–45 degree legend; steep values saturate at 45.
        stops = np.array([[232, 244, 197], [95, 184, 125], [21, 105, 85], [126, 84, 39]])
        t = np.nan_to_num(np.clip(dst / 45, 0, 1)) * 3
        low = np.floor(t).astype(int)
        high = np.minimum(low + 1, 3)
        rgb = stops[low] * (1 - (t-low)[..., None]) + stops[high] * (t-low)[..., None]
    else:
        if not np.isin(dst[valid], [1, 2, 3]).all():
            raise ValueError('Screening raster: encode 1=gentle (<5°), 2=moderate (5–15°), 3=steep (>15°). Other values must be NoData.')
        palette = np.array([[0, 0, 0], [35, 139, 69], [161, 217, 155], [244, 216, 171]])
        rgb = palette[np.nan_to_num(dst).astype(int)]
    rgba = np.dstack([rgb.astype(np.uint8), valid.astype(np.uint8) * 200])
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format='PNG')
    return dict(sourcetype='image', source='data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode(),
                coordinates=[[west, north], [east, north], [east, south], [west, south]], opacity=0.80)


def load_gis_bundle(paths):
    bundle, messages = {}, []
    for kind, path in paths.items():
        if not path:
            continue
        try:
            bundle[kind] = read_geojson(path, kind) if kind in ['catchments', 'flows', 'stations'] else raster_layer(path, kind)
            messages.append(f'{kind}: loaded')
        except Exception as exc:
            messages.append(f'{kind}: REJECTED — {exc}')
    bundle['status'] = '; '.join(messages) if messages else 'No GIS files loaded. Demonstration coordinates only.'
    return bundle


# Original terrain-analysis geometry is embedded: Colab needs no local GIS files.
# Boundaries are DEM modeled, not official basins or surveyed sampling stations.
BUILTIN_GIS = json.loads(r'''{"catchments":{"type":"FeatureCollection","features":[{"type":"Feature","properties":{"site_id":"B01"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.80791666666666,13.138194444444443],[100.80791666666666,13.138472222222221],[100.80791666666666,13.13875],[100.80819444444444,13.13875],[100.80847222222222,13.13875],[100.80847222222222,13.139027777777777],[100.80847222222222,13.139305555555554],[100.80847222222222,13.139583333333333],[100.80875,13.139583333333333],[100.80875,13.13986111111111],[100.80875,13.140138888888888],[100.80875,13.140416666666665],[100.80875,13.140694444444444],[100.80847222222222,13.140694444444444],[100.80847222222222,13.14097222222222],[100.80819444444444,13.14097222222222],[100.80819444444444,13.14125],[100.80791666666666,13.14125],[100.80763888888887,13.14125],[100.8073611111111,13.14125],[100.8073611111111,13.141527777777776],[100.80708333333332,13.141527777777776],[100.80708333333332,13.141805555555555],[100.80680555555554,13.141805555555555],[100.80680555555554,13.142083333333332],[100.80708333333332,13.142083333333332],[100.8073611111111,13.142083333333332],[100.8073611111111,13.141805555555555],[100.80763888888887,13.141805555555555],[100.80791666666666,13.141805555555555],[100.80791666666666,13.142083333333332],[100.80819444444444,13.142083333333332],[100.80819444444444,13.141805555555555],[100.80847222222222,13.141805555555555],[100.80875,13.141805555555555],[100.80902777777777,13.141805555555555],[100.80902777777777,13.141527777777776],[100.80902777777777,13.14125],[100.80930555555555,13.14125],[100.80958333333332,13.14125],[100.80958333333332,13.141527777777776],[100.8098611111111,13.141527777777776],[100.8098611111111,13.141805555555555],[100.81013888888889,13.141805555555555],[100.81041666666665,13.141805555555555],[100.81041666666665,13.142083333333332],[100.81069444444444,13.142083333333332],[100.81069444444444,13.142361111111109],[100.81069444444444,13.142638888888888],[100.81041666666665,13.142638888888888],[100.81041666666665,13.142916666666665],[100.81041666666665,13.143194444444443],[100.81041666666665,13.14347222222222],[100.81041666666665,13.14375],[100.81041666666665,13.144027777777776],[100.81069444444444,13.144027777777776],[100.81069444444444,13.144305555555555],[100.81097222222222,13.144305555555555],[100.81097222222222,13.144583333333332],[100.81125,13.144583333333332],[100.81152777777777,13.144583333333332],[100.81152777777777,13.14486111111111],[100.81180555555555,13.14486111111111],[100.81180555555555,13.144583333333332],[100.81208333333332,13.144583333333332],[100.81208333333332,13.144305555555555],[100.8123611111111,13.144305555555555],[100.8123611111111,13.144027777777776],[100.8123611111111,13.14375],[100.8123611111111,13.14347222222222],[100.81208333333332,13.14347222222222],[100.81180555555555,13.14347222222222],[100.81180555555555,13.143194444444443],[100.81208333333332,13.143194444444443],[100.81208333333332,13.142916666666665],[100.8123611111111,13.142916666666665],[100.8123611111111,13.142638888888888],[100.8123611111111,13.142361111111109],[100.8123611111111,13.142083333333332],[100.81263888888888,13.142083333333332],[100.81263888888888,13.141805555555555],[100.81291666666665,13.141805555555555],[100.81291666666665,13.141527777777776],[100.81319444444443,13.141527777777776],[100.81319444444443,13.14125],[100.81319444444443,13.14097222222222],[100.81347222222222,13.14097222222222],[100.81347222222222,13.140694444444444],[100.81375,13.140694444444444],[100.81402777777777,13.140694444444444],[100.81430555555555,13.140694444444444],[100.81430555555555,13.140416666666665],[100.81458333333333,13.140416666666665],[100.8148611111111,13.140416666666665],[100.81513888888888,13.140416666666665],[100.81513888888888,13.140138888888888],[100.81541666666666,13.140138888888888],[100.81541666666666,13.13986111111111],[100.81513888888888,13.13986111111111],[100.81513888888888,13.139583333333333],[100.81513888888888,13.139305555555554],[100.8148611111111,13.139305555555554],[100.81458333333333,13.139305555555554],[100.81430555555555,13.139305555555554],[100.81402777777777,13.139305555555554],[100.81375,13.139305555555554],[100.81347222222222,13.139305555555554],[100.81319444444443,13.139305555555554],[100.81319444444443,13.139027777777777],[100.81291666666665,13.139027777777777],[100.81291666666665,13.13875],[100.81263888888888,13.13875],[100.81263888888888,13.138472222222221],[100.8123611111111,13.138472222222221],[100.8123611111111,13.138194444444443],[100.8123611111111,13.137916666666666],[100.81208333333332,13.137916666666666],[100.81208333333332,13.137638888888887],[100.81208333333332,13.13736111111111],[100.81180555555555,13.13736111111111],[100.81152777777777,13.13736111111111],[100.81152777777777,13.137638888888887],[100.81125,13.137638888888887],[100.81125,13.137916666666666],[100.81097222222222,13.137916666666666],[100.81097222222222,13.138194444444443],[100.81069444444444,13.138194444444443],[100.81041666666665,13.138194444444443],[100.81041666666665,13.137916666666666],[100.81041666666665,13.137638888888887],[100.81013888888889,13.137638888888887],[100.81013888888889,13.13736111111111],[100.8098611111111,13.13736111111111],[100.80958333333332,13.13736111111111],[100.80930555555555,13.13736111111111],[100.80902777777777,13.13736111111111],[100.80902777777777,13.137638888888887],[100.80875,13.137638888888887],[100.80847222222222,13.137638888888887],[100.80819444444444,13.137638888888887],[100.80791666666666,13.137638888888887],[100.80791666666666,13.137916666666666],[100.80791666666666,13.138194444444443]]],[[[100.80652777777777,13.142361111111109],[100.80680555555554,13.142361111111109],[100.80680555555554,13.142083333333332],[100.80652777777777,13.142083333333332],[100.80652777777777,13.142361111111109]]]]}},{"type":"Feature","properties":{"site_id":"B02"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.80319444444444,13.159305555555553],[100.80291666666666,13.159305555555553],[100.80291666666666,13.159583333333332],[100.80291666666666,13.159861111111109],[100.80263888888888,13.159861111111109],[100.80263888888888,13.160138888888888],[100.80263888888888,13.160416666666665],[100.80263888888888,13.160694444444443],[100.80263888888888,13.16097222222222],[100.80291666666666,13.16097222222222],[100.80291666666666,13.16125],[100.80319444444444,13.16125],[100.80319444444444,13.161527777777776],[100.80347222222221,13.161527777777776],[100.80347222222221,13.161805555555555],[100.80375,13.161805555555555],[100.80375,13.162083333333332],[100.80402777777778,13.162083333333332],[100.80430555555554,13.162083333333332],[100.80430555555554,13.16236111111111],[100.80458333333333,13.16236111111111],[100.80486111111111,13.16236111111111],[100.80486111111111,13.162083333333332],[100.80513888888888,13.162083333333332],[100.80541666666666,13.162083333333332],[100.80569444444444,13.162083333333332],[100.80597222222221,13.162083333333332],[100.80625,13.162083333333332],[100.80625,13.161805555555555],[100.80652777777777,13.161805555555555],[100.80652777777777,13.161527777777776],[100.80652777777777,13.16125],[100.80652777777777,13.16097222222222],[100.80680555555554,13.16097222222222],[100.80708333333332,13.16097222222222],[100.8073611111111,13.16097222222222],[100.80763888888887,13.16097222222222],[100.80763888888887,13.160694444444443],[100.80791666666666,13.160694444444443],[100.80819444444444,13.160694444444443],[100.80819444444444,13.160416666666665],[100.80847222222222,13.160416666666665],[100.80847222222222,13.160138888888888],[100.80875,13.160138888888888],[100.80902777777777,13.160138888888888],[100.80902777777777,13.160416666666665],[100.80930555555555,13.160416666666665],[100.80958333333332,13.160416666666665],[100.80958333333332,13.160138888888888],[100.80958333333332,13.159861111111109],[100.80958333333332,13.159583333333332],[100.80958333333332,13.159305555555553],[100.80958333333332,13.159027777777776],[100.80958333333332,13.15875],[100.80958333333332,13.15847222222222],[100.8098611111111,13.15847222222222],[100.8098611111111,13.158194444444444],[100.80958333333332,13.158194444444444],[100.80958333333332,13.157916666666665],[100.80930555555555,13.157916666666665],[100.80930555555555,13.157638888888888],[100.80902777777777,13.157638888888888],[100.80875,13.157638888888888],[100.80875,13.15736111111111],[100.80847222222222,13.15736111111111],[100.80847222222222,13.157083333333333],[100.80819444444444,13.157083333333333],[100.80819444444444,13.156805555555554],[100.80791666666666,13.156805555555554],[100.80763888888887,13.156805555555554],[100.80763888888887,13.156527777777777],[100.8073611111111,13.156527777777777],[100.80708333333332,13.156527777777777],[100.80680555555554,13.156527777777777],[100.80652777777777,13.156527777777777],[100.80652777777777,13.156805555555554],[100.80625,13.156805555555554],[100.80597222222221,13.156805555555554],[100.80569444444444,13.156805555555554],[100.80569444444444,13.157083333333333],[100.80541666666666,13.157083333333333],[100.80513888888888,13.157083333333333],[100.80513888888888,13.15736111111111],[100.80486111111111,13.15736111111111],[100.80458333333333,13.15736111111111],[100.80430555555554,13.15736111111111],[100.80430555555554,13.157638888888888],[100.80402777777778,13.157638888888888],[100.80402777777778,13.157916666666665],[100.80402777777778,13.158194444444444],[100.80375,13.158194444444444],[100.80375,13.15847222222222],[100.80347222222221,13.15847222222222],[100.80347222222221,13.15875],[100.80319444444444,13.15875],[100.80319444444444,13.159027777777776],[100.80319444444444,13.159305555555553]]],[[[100.80958333333332,13.160694444444443],[100.8098611111111,13.160694444444443],[100.8098611111111,13.160416666666665],[100.80958333333332,13.160416666666665],[100.80958333333332,13.160694444444443]]]]}},{"type":"Feature","properties":{"site_id":"B03"},"geometry":{"type":"Polygon","coordinates":[[[100.81041666666665,13.134583333333332],[100.81013888888889,13.134583333333332],[100.8098611111111,13.134583333333332],[100.8098611111111,13.13486111111111],[100.8098611111111,13.135138888888887],[100.81013888888889,13.135138888888887],[100.81041666666665,13.135138888888887],[100.81069444444444,13.135138888888887],[100.81069444444444,13.135416666666666],[100.81069444444444,13.135694444444443],[100.81069444444444,13.135972222222222],[100.81069444444444,13.13625],[100.81069444444444,13.136527777777776],[100.81069444444444,13.136805555555554],[100.81069444444444,13.137083333333331],[100.81041666666665,13.137083333333331],[100.81013888888889,13.137083333333331],[100.81013888888889,13.13736111111111],[100.81013888888889,13.137638888888887],[100.81041666666665,13.137638888888887],[100.81041666666665,13.137916666666666],[100.81041666666665,13.138194444444443],[100.81069444444444,13.138194444444443],[100.81097222222222,13.138194444444443],[100.81097222222222,13.137916666666666],[100.81125,13.137916666666666],[100.81125,13.137638888888887],[100.81152777777777,13.137638888888887],[100.81152777777777,13.13736111111111],[100.81180555555555,13.13736111111111],[100.81208333333332,13.13736111111111],[100.81208333333332,13.137638888888887],[100.81208333333332,13.137916666666666],[100.8123611111111,13.137916666666666],[100.8123611111111,13.138194444444443],[100.8123611111111,13.138472222222221],[100.81263888888888,13.138472222222221],[100.81263888888888,13.13875],[100.81291666666665,13.13875],[100.81291666666665,13.139027777777777],[100.81319444444443,13.139027777777777],[100.81319444444443,13.139305555555554],[100.81347222222222,13.139305555555554],[100.81375,13.139305555555554],[100.81402777777777,13.139305555555554],[100.81430555555555,13.139305555555554],[100.81458333333333,13.139305555555554],[100.8148611111111,13.139305555555554],[100.8148611111111,13.139027777777777],[100.8148611111111,13.13875],[100.8148611111111,13.138472222222221],[100.8148611111111,13.138194444444443],[100.8148611111111,13.137916666666666],[100.81513888888888,13.137916666666666],[100.81513888888888,13.137638888888887],[100.81513888888888,13.13736111111111],[100.8148611111111,13.13736111111111],[100.8148611111111,13.137083333333331],[100.8148611111111,13.136805555555554],[100.8148611111111,13.136527777777776],[100.81513888888888,13.136527777777776],[100.81513888888888,13.13625],[100.81541666666666,13.13625],[100.81569444444443,13.13625],[100.81569444444443,13.135972222222222],[100.81569444444443,13.135694444444443],[100.81597222222221,13.135694444444443],[100.81597222222221,13.135416666666666],[100.81625,13.135416666666666],[100.81625,13.135138888888887],[100.81652777777776,13.135138888888887],[100.81652777777776,13.13486111111111],[100.81652777777776,13.134583333333332],[100.81652777777776,13.134305555555555],[100.81652777777776,13.134027777777776],[100.81680555555555,13.134027777777776],[100.81680555555555,13.13375],[100.81652777777776,13.13375],[100.81625,13.13375],[100.81625,13.134027777777776],[100.81597222222221,13.134027777777776],[100.81569444444443,13.134027777777776],[100.81541666666666,13.134027777777776],[100.81513888888888,13.134027777777776],[100.81513888888888,13.13375],[100.8148611111111,13.13375],[100.8148611111111,13.13347222222222],[100.81458333333333,13.13347222222222],[100.81430555555555,13.13347222222222],[100.81430555555555,13.133194444444444],[100.81402777777777,13.133194444444444],[100.81375,13.133194444444444],[100.81347222222222,13.133194444444444],[100.81347222222222,13.132916666666665],[100.81319444444443,13.132916666666665],[100.81291666666665,13.132916666666665],[100.81291666666665,13.133194444444444],[100.81263888888888,13.133194444444444],[100.8123611111111,13.133194444444444],[100.8123611111111,13.13347222222222],[100.81208333333332,13.13347222222222],[100.81180555555555,13.13347222222222],[100.81180555555555,13.13375],[100.81180555555555,13.134027777777776],[100.81152777777777,13.134027777777776],[100.81125,13.134027777777776],[100.81097222222222,13.134027777777776],[100.81097222222222,13.134305555555555],[100.81069444444444,13.134305555555555],[100.81041666666665,13.134305555555555],[100.81041666666665,13.134583333333332]]]}},{"type":"Feature","properties":{"site_id":"B04"},"geometry":{"type":"Polygon","coordinates":[[[100.80125,13.164305555555554],[100.80125,13.164583333333333],[100.80125,13.16486111111111],[100.80125,13.165138888888887],[100.80125,13.165416666666665],[100.80125,13.165694444444442],[100.80125,13.165972222222221],[100.80125,13.16625],[100.80125,13.166527777777777],[100.80125,13.166805555555554],[100.80125,13.167083333333332],[100.80152777777776,13.167083333333332],[100.80152777777776,13.16736111111111],[100.80152777777776,13.167638888888888],[100.80152777777776,13.167916666666665],[100.80180555555555,13.167916666666665],[100.80180555555555,13.168194444444444],[100.80208333333333,13.168194444444444],[100.80208333333333,13.167916666666665],[100.80208333333333,13.167638888888888],[100.80208333333333,13.16736111111111],[100.80208333333333,13.167083333333332],[100.8023611111111,13.167083333333332],[100.8023611111111,13.166805555555554],[100.80263888888888,13.166805555555554],[100.80263888888888,13.166527777777777],[100.80291666666666,13.166527777777777],[100.80291666666666,13.16625],[100.80319444444444,13.16625],[100.80347222222221,13.16625],[100.80375,13.16625],[100.80375,13.165972222222221],[100.80402777777778,13.165972222222221],[100.80402777777778,13.165694444444442],[100.80430555555554,13.165694444444442],[100.80430555555554,13.165416666666665],[100.80402777777778,13.165416666666665],[100.80402777777778,13.165138888888887],[100.80430555555554,13.165138888888887],[100.80458333333333,13.165138888888887],[100.80458333333333,13.16486111111111],[100.80458333333333,13.164583333333333],[100.80486111111111,13.164583333333333],[100.80486111111111,13.164305555555554],[100.80513888888888,13.164305555555554],[100.80513888888888,13.164027777777777],[100.80513888888888,13.16375],[100.80541666666666,13.16375],[100.80569444444444,13.16375],[100.80597222222221,13.16375],[100.80625,13.16375],[100.80625,13.164027777777777],[100.80652777777777,13.164027777777777],[100.80680555555554,13.164027777777777],[100.80680555555554,13.16375],[100.80708333333332,13.16375],[100.8073611111111,13.16375],[100.8073611111111,13.163472222222222],[100.8073611111111,13.163194444444443],[100.80763888888887,13.163194444444443],[100.80763888888887,13.162916666666666],[100.80763888888887,13.162638888888887],[100.80791666666666,13.162638888888887],[100.80791666666666,13.16236111111111],[100.80819444444444,13.16236111111111],[100.80847222222222,13.16236111111111],[100.80847222222222,13.162083333333332],[100.80875,13.162083333333332],[100.80875,13.161805555555555],[100.80847222222222,13.161805555555555],[100.80847222222222,13.161527777777776],[100.80819444444444,13.161527777777776],[100.80791666666666,13.161527777777776],[100.80763888888887,13.161527777777776],[100.80763888888887,13.16125],[100.8073611111111,13.16125],[100.80708333333332,13.16125],[100.80680555555554,13.16125],[100.80680555555554,13.16097222222222],[100.80652777777777,13.16097222222222],[100.80652777777777,13.16125],[100.80652777777777,13.161527777777776],[100.80652777777777,13.161805555555555],[100.80625,13.161805555555555],[100.80625,13.162083333333332],[100.80597222222221,13.162083333333332],[100.80569444444444,13.162083333333332],[100.80541666666666,13.162083333333332],[100.80513888888888,13.162083333333332],[100.80486111111111,13.162083333333332],[100.80486111111111,13.16236111111111],[100.80458333333333,13.16236111111111],[100.80430555555554,13.16236111111111],[100.80430555555554,13.162083333333332],[100.80402777777778,13.162083333333332],[100.80375,13.162083333333332],[100.80375,13.161805555555555],[100.80347222222221,13.161805555555555],[100.80347222222221,13.161527777777776],[100.80319444444444,13.161527777777776],[100.80319444444444,13.16125],[100.80291666666666,13.16125],[100.80291666666666,13.16097222222222],[100.80263888888888,13.16097222222222],[100.8023611111111,13.16097222222222],[100.8023611111111,13.16125],[100.8023611111111,13.161527777777776],[100.8023611111111,13.161805555555555],[100.80208333333333,13.161805555555555],[100.80208333333333,13.162083333333332],[100.80208333333333,13.16236111111111],[100.80208333333333,13.162638888888887],[100.80208333333333,13.162916666666666],[100.80208333333333,13.163194444444443],[100.80180555555555,13.163194444444443],[100.80180555555555,13.163472222222222],[100.80152777777776,13.163472222222222],[100.80152777777776,13.16375],[100.80152777777776,13.164027777777777],[100.80125,13.164027777777777],[100.80125,13.164305555555554]]]}},{"type":"Feature","properties":{"site_id":"B05"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.81208333333332,13.143194444444443],[100.81180555555555,13.143194444444443],[100.81180555555555,13.14347222222222],[100.81208333333332,13.14347222222222],[100.8123611111111,13.14347222222222],[100.8123611111111,13.14375],[100.8123611111111,13.144027777777776],[100.8123611111111,13.144305555555555],[100.81208333333332,13.144305555555555],[100.81208333333332,13.144583333333332],[100.81180555555555,13.144583333333332],[100.81180555555555,13.14486111111111],[100.81180555555555,13.145138888888887],[100.81180555555555,13.145416666666666],[100.81180555555555,13.145694444444443],[100.81152777777777,13.145694444444443],[100.81152777777777,13.145972222222222],[100.81125,13.145972222222222],[100.81125,13.14625],[100.81097222222222,13.14625],[100.81097222222222,13.146527777777777],[100.81125,13.146527777777777],[100.81125,13.146805555555554],[100.81152777777777,13.146805555555554],[100.81180555555555,13.146805555555554],[100.81208333333332,13.146805555555554],[100.8123611111111,13.146805555555554],[100.81263888888888,13.146805555555554],[100.81263888888888,13.146527777777777],[100.81291666666665,13.146527777777777],[100.81291666666665,13.14625],[100.81319444444443,13.14625],[100.81319444444443,13.145972222222222],[100.81319444444443,13.145694444444443],[100.81319444444443,13.145416666666666],[100.81319444444443,13.145138888888887],[100.81319444444443,13.14486111111111],[100.81347222222222,13.14486111111111],[100.81347222222222,13.144583333333332],[100.81375,13.144583333333332],[100.81375,13.14486111111111],[100.81402777777777,13.14486111111111],[100.81402777777777,13.144583333333332],[100.81430555555555,13.144583333333332],[100.81458333333333,13.144583333333332],[100.8148611111111,13.144583333333332],[100.8148611111111,13.144305555555555],[100.8148611111111,13.144027777777776],[100.81513888888888,13.144027777777776],[100.81513888888888,13.14375],[100.81513888888888,13.14347222222222],[100.81513888888888,13.143194444444443],[100.81541666666666,13.143194444444443],[100.81541666666666,13.142916666666665],[100.81569444444443,13.142916666666665],[100.81569444444443,13.142638888888888],[100.81597222222221,13.142638888888888],[100.81597222222221,13.142361111111109],[100.81625,13.142361111111109],[100.81652777777776,13.142361111111109],[100.81680555555555,13.142361111111109],[100.81708333333333,13.142361111111109],[100.8173611111111,13.142361111111109],[100.81763888888888,13.142361111111109],[100.81763888888888,13.142083333333332],[100.81763888888888,13.141805555555555],[100.8173611111111,13.141805555555555],[100.8173611111111,13.141527777777776],[100.81708333333333,13.141527777777776],[100.81708333333333,13.14125],[100.81680555555555,13.14125],[100.81652777777776,13.14125],[100.81652777777776,13.14097222222222],[100.81625,13.14097222222222],[100.81625,13.140694444444444],[100.81597222222221,13.140694444444444],[100.81569444444443,13.140694444444444],[100.81541666666666,13.140694444444444],[100.81513888888888,13.140694444444444],[100.81513888888888,13.140416666666665],[100.8148611111111,13.140416666666665],[100.81458333333333,13.140416666666665],[100.81430555555555,13.140416666666665],[100.81430555555555,13.140694444444444],[100.81402777777777,13.140694444444444],[100.81375,13.140694444444444],[100.81347222222222,13.140694444444444],[100.81347222222222,13.14097222222222],[100.81319444444443,13.14097222222222],[100.81319444444443,13.14125],[100.81319444444443,13.141527777777776],[100.81291666666665,13.141527777777776],[100.81291666666665,13.141805555555555],[100.81263888888888,13.141805555555555],[100.81263888888888,13.142083333333332],[100.8123611111111,13.142083333333332],[100.8123611111111,13.142361111111109],[100.8123611111111,13.142638888888888],[100.8123611111111,13.142916666666665],[100.81208333333332,13.142916666666665],[100.81208333333332,13.143194444444443]]],[[[100.81763888888888,13.142361111111109],[100.81763888888888,13.142638888888888],[100.81791666666666,13.142638888888888],[100.81791666666666,13.142361111111109],[100.81763888888888,13.142361111111109]]]]}},{"type":"Feature","properties":{"site_id":"B06"},"geometry":{"type":"Polygon","coordinates":[[[100.80263888888888,13.166805555555554],[100.8023611111111,13.166805555555554],[100.8023611111111,13.167083333333332],[100.80208333333333,13.167083333333332],[100.80208333333333,13.16736111111111],[100.80208333333333,13.167638888888888],[100.80208333333333,13.167916666666665],[100.80208333333333,13.168194444444444],[100.80208333333333,13.16847222222222],[100.8023611111111,13.16847222222222],[100.8023611111111,13.16875],[100.8023611111111,13.169027777777776],[100.80263888888888,13.169027777777776],[100.80263888888888,13.169305555555555],[100.80263888888888,13.169583333333332],[100.80291666666666,13.169583333333332],[100.80319444444444,13.169583333333332],[100.80319444444444,13.16986111111111],[100.80347222222221,13.16986111111111],[100.80347222222221,13.170138888888888],[100.80375,13.170138888888888],[100.80402777777778,13.170138888888888],[100.80402777777778,13.170416666666664],[100.80430555555554,13.170416666666664],[100.80430555555554,13.170694444444443],[100.80458333333333,13.170694444444443],[100.80486111111111,13.170694444444443],[100.80486111111111,13.17097222222222],[100.80513888888888,13.17097222222222],[100.80513888888888,13.170694444444443],[100.80541666666666,13.170694444444443],[100.80541666666666,13.170416666666664],[100.80541666666666,13.170138888888888],[100.80541666666666,13.16986111111111],[100.80541666666666,13.169583333333332],[100.80541666666666,13.169305555555555],[100.80513888888888,13.169305555555555],[100.80513888888888,13.169027777777776],[100.80486111111111,13.169027777777776],[100.80486111111111,13.16875],[100.80486111111111,13.16847222222222],[100.80486111111111,13.168194444444444],[100.80486111111111,13.167916666666665],[100.80486111111111,13.167638888888888],[100.80486111111111,13.16736111111111],[100.80486111111111,13.167083333333332],[100.80486111111111,13.166805555555554],[100.80486111111111,13.166527777777777],[100.80486111111111,13.16625],[100.80513888888888,13.16625],[100.80513888888888,13.166527777777777],[100.80541666666666,13.166527777777777],[100.80569444444444,13.166527777777777],[100.80569444444444,13.16625],[100.80597222222221,13.16625],[100.80597222222221,13.165972222222221],[100.80625,13.165972222222221],[100.80652777777777,13.165972222222221],[100.80652777777777,13.165694444444442],[100.80680555555554,13.165694444444442],[100.80680555555554,13.165416666666665],[100.80708333333332,13.165416666666665],[100.8073611111111,13.165416666666665],[100.80763888888887,13.165416666666665],[100.80791666666666,13.165416666666665],[100.80791666666666,13.165138888888887],[100.80763888888887,13.165138888888887],[100.80763888888887,13.16486111111111],[100.8073611111111,13.16486111111111],[100.8073611111111,13.164583333333333],[100.80708333333332,13.164583333333333],[100.80708333333332,13.164305555555554],[100.80680555555554,13.164305555555554],[100.80680555555554,13.164027777777777],[100.80652777777777,13.164027777777777],[100.80625,13.164027777777777],[100.80625,13.16375],[100.80597222222221,13.16375],[100.80569444444444,13.16375],[100.80541666666666,13.16375],[100.80513888888888,13.16375],[100.80513888888888,13.164027777777777],[100.80513888888888,13.164305555555554],[100.80486111111111,13.164305555555554],[100.80486111111111,13.164583333333333],[100.80458333333333,13.164583333333333],[100.80458333333333,13.16486111111111],[100.80458333333333,13.165138888888887],[100.80430555555554,13.165138888888887],[100.80402777777778,13.165138888888887],[100.80402777777778,13.165416666666665],[100.80430555555554,13.165416666666665],[100.80430555555554,13.165694444444442],[100.80402777777778,13.165694444444442],[100.80402777777778,13.165972222222221],[100.80375,13.165972222222221],[100.80375,13.16625],[100.80347222222221,13.16625],[100.80319444444444,13.16625],[100.80291666666666,13.16625],[100.80291666666666,13.166527777777777],[100.80263888888888,13.166527777777777],[100.80263888888888,13.166805555555554]]]}},{"type":"Feature","properties":{"site_id":"B07"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.80847222222222,13.12875],[100.80875,13.12875],[100.80875,13.129027777777777],[100.80875,13.129305555555554],[100.80875,13.129583333333333],[100.80875,13.12986111111111],[100.80875,13.130138888888888],[100.80875,13.130416666666665],[100.80875,13.130694444444442],[100.80847222222222,13.130694444444442],[100.80819444444444,13.130694444444442],[100.80819444444444,13.130972222222221],[100.80819444444444,13.13125],[100.80819444444444,13.131527777777777],[100.80819444444444,13.131805555555554],[100.80847222222222,13.131805555555554],[100.80847222222222,13.132083333333332],[100.80847222222222,13.13236111111111],[100.80875,13.13236111111111],[100.80875,13.132638888888888],[100.80902777777777,13.132638888888888],[100.80902777777777,13.132916666666665],[100.80902777777777,13.133194444444444],[100.80902777777777,13.13347222222222],[100.80902777777777,13.13375],[100.80902777777777,13.134027777777776],[100.80930555555555,13.134027777777776],[100.80930555555555,13.13375],[100.80958333333332,13.13375],[100.80958333333332,13.13347222222222],[100.8098611111111,13.13347222222222],[100.81013888888889,13.13347222222222],[100.81013888888889,13.133194444444444],[100.81041666666665,13.133194444444444],[100.81069444444444,13.133194444444444],[100.81069444444444,13.132916666666665],[100.81097222222222,13.132916666666665],[100.81125,13.132916666666665],[100.81152777777777,13.132916666666665],[100.81152777777777,13.132638888888888],[100.81152777777777,13.13236111111111],[100.81152777777777,13.132083333333332],[100.81180555555555,13.132083333333332],[100.81180555555555,13.131805555555554],[100.81208333333332,13.131805555555554],[100.81208333333332,13.131527777777777],[100.81208333333332,13.13125],[100.81208333333332,13.130972222222221],[100.81208333333332,13.130694444444442],[100.81180555555555,13.130694444444442],[100.81152777777777,13.130694444444442],[100.81152777777777,13.130416666666665],[100.81125,13.130416666666665],[100.81125,13.130138888888888],[100.81097222222222,13.130138888888888],[100.81097222222222,13.12986111111111],[100.81097222222222,13.129583333333333],[100.81069444444444,13.129583333333333],[100.81069444444444,13.129305555555554],[100.81069444444444,13.129027777777777],[100.81069444444444,13.12875],[100.81097222222222,13.12875],[100.81097222222222,13.128472222222221],[100.81097222222222,13.128194444444443],[100.81097222222222,13.127916666666666],[100.81097222222222,13.127638888888887],[100.81097222222222,13.12736111111111],[100.81069444444444,13.12736111111111],[100.81041666666665,13.12736111111111],[100.81013888888889,13.12736111111111],[100.81013888888889,13.127638888888887],[100.8098611111111,13.127638888888887],[100.80958333333332,13.127638888888887],[100.80930555555555,13.127638888888887],[100.80902777777777,13.127638888888887],[100.80875,13.127638888888887],[100.80847222222222,13.127638888888887],[100.80847222222222,13.127916666666666],[100.80847222222222,13.128194444444443],[100.80847222222222,13.128472222222221],[100.80847222222222,13.12875]]],[[[100.81097222222222,13.127083333333331],[100.81097222222222,13.12736111111111],[100.81125,13.12736111111111],[100.81125,13.127083333333331],[100.81097222222222,13.127083333333331]]]]}},{"type":"Feature","properties":{"site_id":"B08"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.80708333333332,13.15125],[100.80708333333332,13.151527777777776],[100.80708333333332,13.151805555555555],[100.80708333333332,13.152083333333332],[100.80708333333332,13.15236111111111],[100.80708333333332,13.152638888888887],[100.8073611111111,13.152638888888887],[100.8073611111111,13.152916666666666],[100.8073611111111,13.153194444444443],[100.8073611111111,13.15347222222222],[100.80763888888887,13.15347222222222],[100.80763888888887,13.15375],[100.80763888888887,13.154027777777776],[100.80791666666666,13.154027777777776],[100.80791666666666,13.154305555555554],[100.80791666666666,13.154583333333331],[100.80791666666666,13.15486111111111],[100.80819444444444,13.15486111111111],[100.80847222222222,13.15486111111111],[100.80875,13.15486111111111],[100.80902777777777,13.15486111111111],[100.80930555555555,13.15486111111111],[100.80930555555555,13.155138888888887],[100.80958333333332,13.155138888888887],[100.8098611111111,13.155138888888887],[100.81013888888889,13.155138888888887],[100.81013888888889,13.155416666666666],[100.81041666666665,13.155416666666666],[100.81069444444444,13.155416666666666],[100.81069444444444,13.155694444444443],[100.81097222222222,13.155694444444443],[100.81097222222222,13.155972222222221],[100.81125,13.155972222222221],[100.81125,13.15625],[100.81152777777777,13.15625],[100.81152777777777,13.156527777777777],[100.81180555555555,13.156527777777777],[100.81208333333332,13.156527777777777],[100.8123611111111,13.156527777777777],[100.8123611111111,13.15625],[100.8123611111111,13.155972222222221],[100.81208333333332,13.155972222222221],[100.81208333333332,13.155694444444443],[100.81208333333332,13.155416666666666],[100.81180555555555,13.155416666666666],[100.81152777777777,13.155416666666666],[100.81152777777777,13.155138888888887],[100.81125,13.155138888888887],[100.81097222222222,13.155138888888887],[100.81097222222222,13.15486111111111],[100.81097222222222,13.154583333333331],[100.81097222222222,13.154305555555554],[100.81069444444444,13.154305555555554],[100.81069444444444,13.154027777777776],[100.81041666666665,13.154027777777776],[100.81041666666665,13.15375],[100.81041666666665,13.15347222222222],[100.81041666666665,13.153194444444443],[100.81041666666665,13.152916666666666],[100.81041666666665,13.152638888888887],[100.81041666666665,13.15236111111111],[100.81041666666665,13.152083333333332],[100.81041666666665,13.151805555555555],[100.81041666666665,13.151527777777776],[100.81013888888889,13.151527777777776],[100.81013888888889,13.15125],[100.8098611111111,13.15125],[100.80958333333332,13.15125],[100.80930555555555,13.15125],[100.80902777777777,13.15125],[100.80875,13.15125],[100.80847222222222,13.15125],[100.80847222222222,13.15097222222222],[100.80819444444444,13.15097222222222],[100.80819444444444,13.150694444444444],[100.80791666666666,13.150694444444444],[100.80763888888887,13.150694444444444],[100.80763888888887,13.150416666666665],[100.8073611111111,13.150416666666665],[100.8073611111111,13.150138888888888],[100.80708333333332,13.150138888888888],[100.80680555555554,13.150138888888888],[100.80652777777777,13.150138888888888],[100.80652777777777,13.150416666666665],[100.80680555555554,13.150416666666665],[100.80680555555554,13.150694444444444],[100.80708333333332,13.150694444444444],[100.80708333333332,13.15097222222222],[100.80708333333332,13.15125]]],[[[100.8123611111111,13.156527777777777],[100.8123611111111,13.156805555555554],[100.81263888888888,13.156805555555554],[100.81263888888888,13.156527777777777],[100.8123611111111,13.156527777777777]]]]}},{"type":"Feature","properties":{"site_id":"B09"},"geometry":{"type":"MultiPolygon","coordinates":[[[[100.80930555555555,13.149583333333332],[100.80902777777777,13.149583333333332],[100.80902777777777,13.14986111111111],[100.80902777777777,13.150138888888888],[100.80902777777777,13.150416666666665],[100.80902777777777,13.150694444444444],[100.80902777777777,13.15097222222222],[100.80902777777777,13.15125],[100.80930555555555,13.15125],[100.80958333333332,13.15125],[100.8098611111111,13.15125],[100.81013888888889,13.15125],[100.81013888888889,13.151527777777776],[100.81041666666665,13.151527777777776],[100.81069444444444,13.151527777777776],[100.81097222222222,13.151527777777776],[100.81097222222222,13.151805555555555],[100.81125,13.151805555555555],[100.81125,13.152083333333332],[100.81152777777777,13.152083333333332],[100.81152777777777,13.15236111111111],[100.81180555555555,13.15236111111111],[100.81180555555555,13.152083333333332],[100.81208333333332,13.152083333333332],[100.81208333333332,13.151805555555555],[100.8123611111111,13.151805555555555],[100.81263888888888,13.151805555555555],[100.81291666666665,13.151805555555555],[100.81319444444443,13.151805555555555],[100.81347222222222,13.151805555555555],[100.81375,13.151805555555555],[100.81375,13.152083333333332],[100.81402777777777,13.152083333333332],[100.81430555555555,13.152083333333332],[100.81430555555555,13.151805555555555],[100.81430555555555,13.151527777777776],[100.81430555555555,13.15125],[100.81430555555555,13.15097222222222],[100.81430555555555,13.150694444444444],[100.81430555555555,13.150416666666665],[100.81430555555555,13.150138888888888],[100.81430555555555,13.14986111111111],[100.81430555555555,13.149583333333332],[100.81402777777777,13.149583333333332],[100.81375,13.149583333333332],[100.81347222222222,13.149583333333332],[100.81347222222222,13.149305555555554],[100.81319444444443,13.149305555555554],[100.81319444444443,13.149027777777777],[100.81291666666665,13.149027777777777],[100.81291666666665,13.14875],[100.81263888888888,13.14875],[100.8123611111111,13.14875],[100.8123611111111,13.149027777777777],[100.81208333333332,13.149027777777777],[100.81180555555555,13.149027777777777],[100.81180555555555,13.14875],[100.81152777777777,13.14875],[100.81125,13.14875],[100.81097222222222,13.14875],[100.81069444444444,13.14875],[100.81041666666665,13.14875],[100.81013888888889,13.14875],[100.81013888888889,13.149027777777777],[100.8098611111111,13.149027777777777],[100.80958333333332,13.149027777777777],[100.80930555555555,13.149027777777777],[100.80930555555555,13.149305555555554],[100.80930555555555,13.149583333333332]]],[[[100.81458333333333,13.15236111111111],[100.81458333333333,13.152083333333332],[100.81430555555555,13.152083333333332],[100.81430555555555,13.15236111111111],[100.81458333333333,13.15236111111111]]],[[[100.81458333333333,13.15236111111111],[100.81458333333333,13.152638888888887],[100.8148611111111,13.152638888888887],[100.8148611111111,13.15236111111111],[100.81458333333333,13.15236111111111]]]]}},{"type":"Feature","properties":{"site_id":"B10"},"geometry":{"type":"Polygon","coordinates":[[[100.80763888888887,13.146527777777777],[100.80763888888887,13.146805555555554],[100.80791666666666,13.146805555555554],[100.80791666666666,13.147083333333333],[100.80819444444444,13.147083333333333],[100.80819444444444,13.14736111111111],[100.80847222222222,13.14736111111111],[100.80847222222222,13.147638888888888],[100.80875,13.147638888888888],[100.80875,13.147916666666665],[100.80875,13.148194444444442],[100.80875,13.148472222222221],[100.80847222222222,13.148472222222221],[100.80847222222222,13.14875],[100.80847222222222,13.149027777777777],[100.80847222222222,13.149305555555554],[100.80819444444444,13.149305555555554],[100.80819444444444,13.149583333333332],[100.80791666666666,13.149583333333332],[100.80791666666666,13.14986111111111],[100.80763888888887,13.14986111111111],[100.80763888888887,13.150138888888888],[100.8073611111111,13.150138888888888],[100.8073611111111,13.150416666666665],[100.80763888888887,13.150416666666665],[100.80763888888887,13.150694444444444],[100.80791666666666,13.150694444444444],[100.80819444444444,13.150694444444444],[100.80819444444444,13.15097222222222],[100.80847222222222,13.15097222222222],[100.80847222222222,13.15125],[100.80875,13.15125],[100.80902777777777,13.15125],[100.80902777777777,13.15097222222222],[100.80902777777777,13.150694444444444],[100.80902777777777,13.150416666666665],[100.80902777777777,13.150138888888888],[100.80902777777777,13.14986111111111],[100.80902777777777,13.149583333333332],[100.80930555555555,13.149583333333332],[100.80930555555555,13.149305555555554],[100.80930555555555,13.149027777777777],[100.80930555555555,13.14875],[100.80930555555555,13.148472222222221],[100.80958333333332,13.148472222222221],[100.80958333333332,13.148194444444442],[100.8098611111111,13.148194444444442],[100.8098611111111,13.147916666666665],[100.81013888888889,13.147916666666665],[100.81041666666665,13.147916666666665],[100.81069444444444,13.147916666666665],[100.81069444444444,13.147638888888888],[100.81097222222222,13.147638888888888],[100.81097222222222,13.14736111111111],[100.81097222222222,13.147083333333333],[100.81125,13.147083333333333],[100.81125,13.146805555555554],[100.81125,13.146527777777777],[100.81097222222222,13.146527777777777],[100.81097222222222,13.14625],[100.81125,13.14625],[100.81125,13.145972222222222],[100.81152777777777,13.145972222222222],[100.81152777777777,13.145694444444443],[100.81180555555555,13.145694444444443],[100.81180555555555,13.145416666666666],[100.81180555555555,13.145138888888887],[100.81180555555555,13.14486111111111],[100.81152777777777,13.14486111111111],[100.81152777777777,13.144583333333332],[100.81125,13.144583333333332],[100.81097222222222,13.144583333333332],[100.81097222222222,13.144305555555555],[100.81069444444444,13.144305555555555],[100.81069444444444,13.144027777777776],[100.81041666666665,13.144027777777776],[100.81013888888889,13.144027777777776],[100.81013888888889,13.144305555555555],[100.8098611111111,13.144305555555555],[100.8098611111111,13.144583333333332],[100.80958333333332,13.144583333333332],[100.80958333333332,13.14486111111111],[100.80958333333332,13.145138888888887],[100.80958333333332,13.145416666666666],[100.80930555555555,13.145416666666666],[100.80930555555555,13.145694444444443],[100.80902777777777,13.145694444444443],[100.80875,13.145694444444443],[100.80847222222222,13.145694444444443],[100.80819444444444,13.145694444444443],[100.80791666666666,13.145694444444443],[100.80763888888887,13.145694444444443],[100.8073611111111,13.145694444444443],[100.80708333333332,13.145694444444443],[100.80680555555554,13.145694444444443],[100.80680555555554,13.145972222222222],[100.80708333333332,13.145972222222222],[100.80708333333332,13.14625],[100.8073611111111,13.14625],[100.8073611111111,13.146527777777777],[100.80763888888887,13.146527777777777]]]}},{"type":"Feature","properties":{"site_id":"B11"},"geometry":{"type":"Polygon","coordinates":[[[100.81069444444444,13.147638888888888],[100.81069444444444,13.147916666666665],[100.81041666666665,13.147916666666665],[100.81013888888889,13.147916666666665],[100.8098611111111,13.147916666666665],[100.8098611111111,13.148194444444442],[100.80958333333332,13.148194444444442],[100.80958333333332,13.148472222222221],[100.80930555555555,13.148472222222221],[100.80930555555555,13.14875],[100.80930555555555,13.149027777777777],[100.80958333333332,13.149027777777777],[100.8098611111111,13.149027777777777],[100.81013888888889,13.149027777777777],[100.81013888888889,13.14875],[100.81041666666665,13.14875],[100.81069444444444,13.14875],[100.81097222222222,13.14875],[100.81125,13.14875],[100.81152777777777,13.14875],[100.81180555555555,13.14875],[100.81180555555555,13.149027777777777],[100.81208333333332,13.149027777777777],[100.8123611111111,13.149027777777777],[100.8123611111111,13.14875],[100.81263888888888,13.14875],[100.81291666666665,13.14875],[100.81291666666665,13.149027777777777],[100.81319444444443,13.149027777777777],[100.81319444444443,13.149305555555554],[100.81347222222222,13.149305555555554],[100.81347222222222,13.149583333333332],[100.81375,13.149583333333332],[100.81402777777777,13.149583333333332],[100.81430555555555,13.149583333333332],[100.81458333333333,13.149583333333332],[100.8148611111111,13.149583333333332],[100.81513888888888,13.149583333333332],[100.81513888888888,13.149305555555554],[100.81541666666666,13.149305555555554],[100.81569444444443,13.149305555555554],[100.81569444444443,13.149027777777777],[100.81597222222221,13.149027777777777],[100.81597222222221,13.149305555555554],[100.81625,13.149305555555554],[100.81652777777776,13.149305555555554],[100.81652777777776,13.149583333333332],[100.81680555555555,13.149583333333332],[100.81680555555555,13.14986111111111],[100.81708333333333,13.14986111111111],[100.81708333333333,13.150138888888888],[100.8173611111111,13.150138888888888],[100.8173611111111,13.14986111111111],[100.8173611111111,13.149583333333332],[100.81763888888888,13.149583333333332],[100.81763888888888,13.149305555555554],[100.81763888888888,13.149027777777777],[100.8173611111111,13.149027777777777],[100.8173611111111,13.14875],[100.81708333333333,13.14875],[100.81680555555555,13.14875],[100.81680555555555,13.148472222222221],[100.81652777777776,13.148472222222221],[100.81652777777776,13.148194444444442],[100.81625,13.148194444444442],[100.81625,13.147916666666665],[100.81597222222221,13.147916666666665],[100.81569444444443,13.147916666666665],[100.81569444444443,13.148194444444442],[100.81541666666666,13.148194444444442],[100.81513888888888,13.148194444444442],[100.8148611111111,13.148194444444442],[100.81458333333333,13.148194444444442],[100.81430555555555,13.148194444444442],[100.81402777777777,13.148194444444442],[100.81375,13.148194444444442],[100.81375,13.148472222222221],[100.81347222222222,13.148472222222221],[100.81319444444443,13.148472222222221],[100.81291666666665,13.148472222222221],[100.81291666666665,13.148194444444442],[100.81291666666665,13.147916666666665],[100.81291666666665,13.147638888888888],[100.81291666666665,13.14736111111111],[100.81263888888888,13.14736111111111],[100.81263888888888,13.147083333333333],[100.81263888888888,13.146805555555554],[100.8123611111111,13.146805555555554],[100.81208333333332,13.146805555555554],[100.81180555555555,13.146805555555554],[100.81152777777777,13.146805555555554],[100.81125,13.146805555555554],[100.81125,13.147083333333333],[100.81097222222222,13.147083333333333],[100.81097222222222,13.14736111111111],[100.81097222222222,13.147638888888888],[100.81069444444444,13.147638888888888]]]}},{"type":"Feature","properties":{"site_id":"B12"},"geometry":{"type":"Polygon","coordinates":[[[100.81319444444443,13.14625],[100.81291666666665,13.14625],[100.81291666666665,13.146527777777777],[100.81263888888888,13.146527777777777],[100.81263888888888,13.146805555555554],[100.81263888888888,13.147083333333333],[100.81263888888888,13.14736111111111],[100.81291666666665,13.14736111111111],[100.81291666666665,13.147638888888888],[100.81291666666665,13.147916666666665],[100.81291666666665,13.148194444444442],[100.81291666666665,13.148472222222221],[100.81319444444443,13.148472222222221],[100.81347222222222,13.148472222222221],[100.81375,13.148472222222221],[100.81375,13.148194444444442],[100.81402777777777,13.148194444444442],[100.81430555555555,13.148194444444442],[100.81458333333333,13.148194444444442],[100.8148611111111,13.148194444444442],[100.81513888888888,13.148194444444442],[100.81541666666666,13.148194444444442],[100.81569444444443,13.148194444444442],[100.81569444444443,13.147916666666665],[100.81597222222221,13.147916666666665],[100.81625,13.147916666666665],[100.81625,13.148194444444442],[100.81652777777776,13.148194444444442],[100.81652777777776,13.148472222222221],[100.81680555555555,13.148472222222221],[100.81680555555555,13.14875],[100.81708333333333,13.14875],[100.8173611111111,13.14875],[100.8173611111111,13.149027777777777],[100.81763888888888,13.149027777777777],[100.81763888888888,13.14875],[100.81763888888888,13.148472222222221],[100.81791666666666,13.148472222222221],[100.81791666666666,13.148194444444442],[100.81763888888888,13.148194444444442],[100.81763888888888,13.147916666666665],[100.81763888888888,13.147638888888888],[100.8173611111111,13.147638888888888],[100.81708333333333,13.147638888888888],[100.81708333333333,13.14736111111111],[100.81680555555555,13.14736111111111],[100.81680555555555,13.147083333333333],[100.81652777777776,13.147083333333333],[100.81652777777776,13.146805555555554],[100.81652777777776,13.146527777777777],[100.81625,13.146527777777777],[100.81625,13.14625],[100.81597222222221,13.14625],[100.81569444444443,13.14625],[100.81541666666666,13.14625],[100.81513888888888,13.14625],[100.81513888888888,13.145972222222222],[100.8148611111111,13.145972222222222],[100.81458333333333,13.145972222222222],[100.81458333333333,13.145694444444443],[100.81430555555555,13.145694444444443],[100.81430555555555,13.145416666666666],[100.81402777777777,13.145416666666666],[100.81402777777777,13.145138888888887],[100.81375,13.145138888888887],[100.81375,13.14486111111111],[100.81347222222222,13.14486111111111],[100.81319444444443,13.14486111111111],[100.81319444444443,13.145138888888887],[100.81319444444443,13.145416666666666],[100.81319444444443,13.145694444444443],[100.81319444444443,13.145972222222222],[100.81319444444443,13.14625]]]}}]},"flows":{"type":"FeatureCollection","features":[{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.801944444444,13.1733333333333],[100.801666666667,13.1733333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.802222222222,13.1733333333333],[100.801944444444,13.1733333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.803611111111,13.1652777777778],[100.803611111111,13.165]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.803611111111,13.165],[100.803888888889,13.1647222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.803888888889,13.1647222222222],[100.803888888889,13.1644444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.803888888889,13.1644444444444],[100.803888888889,13.1641666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.803888888889,13.1641666666667],[100.804166666667,13.1638888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804166666667,13.1638888888889],[100.804166666667,13.1636111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804166666667,13.1636111111111],[100.804166666667,13.1633333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804166666667,13.1633333333333],[100.804166666667,13.1630555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804166666667,13.1630555555556],[100.804444444444,13.1627777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804166666667,13.1625],[100.804444444444,13.1627777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1677777777778],[100.804444444444,13.1675]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1675],[100.804444444444,13.1672222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1672222222222],[100.804722222222,13.1669444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1663888888889],[100.804444444444,13.1661111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1661111111111],[100.804444444444,13.1658333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1658333333333],[100.804444444444,13.1655555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1655555555556],[100.804722222222,13.1652777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804444444444,13.1627777777778],[100.804722222222,13.1627777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804722222222,13.1669444444444],[100.804722222222,13.1666666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804722222222,13.1666666666667],[100.804444444444,13.1663888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804722222222,13.1652777777778],[100.805,13.165]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.804722222222,13.1627777777778],[100.805,13.1625]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805,13.165],[100.805,13.1647222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805,13.1647222222222],[100.805277777778,13.1644444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805,13.1625],[100.805277777778,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805277777778,13.1644444444444],[100.805555555556,13.1641666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805277777778,13.1622222222222],[100.805555555556,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805555555556,13.1641666666667],[100.805833333333,13.1638888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805555555556,13.1622222222222],[100.805833333333,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805555555556,13.1608333333333],[100.805833333333,13.1608333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805555555556,13.1544444444444],[100.805277777778,13.1547222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805833333333,13.1638888888889],[100.806111111111,13.1638888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805833333333,13.1622222222222],[100.806111111111,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805833333333,13.1608333333333],[100.806111111111,13.1608333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805833333333,13.16],[100.806111111111,13.1602777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.805833333333,13.1541666666667],[100.805555555556,13.1544444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806111111111,13.1638888888889],[100.806388888889,13.1641666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806111111111,13.1622222222222],[100.806388888889,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806111111111,13.1608333333333],[100.806388888889,13.1605555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806111111111,13.1602777777778],[100.806388888889,13.1602777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806111111111,13.1538888888889],[100.805833333333,13.1541666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806388888889,13.1641666666667],[100.806666666667,13.1644444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806388888889,13.1622222222222],[100.806666666667,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806388888889,13.1605555555556],[100.806388888889,13.1602777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806388888889,13.1602777777778],[100.806666666667,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.1647222222222],[100.806944444444,13.165]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.1644444444444],[100.806666666667,13.1647222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.1622222222222],[100.806944444444,13.1625]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.16],[100.806666666667,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.1597222222222],[100.806944444444,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806666666667,13.1594444444444],[100.806944444444,13.1594444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806944444444,13.165],[100.807222222222,13.1652777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806944444444,13.1625],[100.807222222222,13.1627777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806944444444,13.1597222222222],[100.807222222222,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806944444444,13.1594444444444],[100.807222222222,13.1594444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.806944444444,13.1419444444444],[100.806666666667,13.1422222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1652777777778],[100.8075,13.1652777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1627777777778],[100.8075,13.1625]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1597222222222],[100.8075,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1594444444444],[100.8075,13.1594444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1458333333333],[100.806944444444,13.1458333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1419444444444],[100.806944444444,13.1419444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807222222222,13.1355555555556],[100.806944444444,13.1352777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1652777777778],[100.807777777778,13.1652777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1625],[100.807777777778,13.1622222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1597222222222],[100.807777777778,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1594444444444],[100.807777777778,13.1597222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1461111111111],[100.807222222222,13.1458333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1416666666667],[100.807222222222,13.1419444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8075,13.1355555555556],[100.807222222222,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807777777778,13.1622222222222],[100.808055555556,13.1619444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807777777778,13.1597222222222],[100.808055555556,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807777777778,13.1461111111111],[100.8075,13.1461111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807777777778,13.1416666666667],[100.8075,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.807777777778,13.1355555555556],[100.8075,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808055555556,13.1619444444444],[100.808333333333,13.1619444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808055555556,13.16],[100.808333333333,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808055555556,13.1461111111111],[100.807777777778,13.1461111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808055555556,13.1416666666667],[100.807777777778,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808055555556,13.1355555555556],[100.807777777778,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1619444444444],[100.808611111111,13.1619444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.16],[100.808611111111,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1533333333333],[100.808611111111,13.1536111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1463888888889],[100.808055555556,13.1461111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1416666666667],[100.808055555556,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1413888888889],[100.808333333333,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808333333333,13.1355555555556],[100.808055555556,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808611111111,13.16],[100.808888888889,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808611111111,13.1536111111111],[100.808888888889,13.1536111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808611111111,13.1463888888889],[100.808333333333,13.1463888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808611111111,13.1411111111111],[100.808333333333,13.1413888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808611111111,13.1358333333333],[100.808333333333,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.16],[100.809166666667,13.16]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.1536111111111],[100.809166666667,13.1536111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.1463888888889],[100.808611111111,13.1463888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.1408333333333],[100.808611111111,13.1411111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.1405555555556],[100.808888888889,13.1408333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.808888888889,13.1402777777778],[100.808888888889,13.1405555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809166666667,13.16],[100.809444444444,13.1602777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809166666667,13.1536111111111],[100.809444444444,13.1538888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809166666667,13.1461111111111],[100.808888888889,13.1463888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809166666667,13.1408333333333],[100.808888888889,13.1408333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809166666667,13.14],[100.808888888889,13.1402777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809444444444,13.1602777777778],[100.809722222222,13.1605555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809444444444,13.1538888888889],[100.809722222222,13.1541666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809444444444,13.1461111111111],[100.809166666667,13.1461111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809444444444,13.1408333333333],[100.809166666667,13.1408333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809722222222,13.1541666666667],[100.81,13.1544444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809722222222,13.1411111111111],[100.809444444444,13.1408333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809722222222,13.1405555555556],[100.809444444444,13.1408333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809722222222,13.1308333333333],[100.809722222222,13.1305555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.809722222222,13.1305555555556],[100.81,13.1302777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1544444444444],[100.810277777778,13.1547222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1411111111111],[100.809722222222,13.1411111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1402777777778],[100.809722222222,13.1405555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1308333333333],[100.81,13.1305555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1305555555556],[100.81,13.1302777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1302777777778],[100.81,13.13]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.13],[100.81,13.1297222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.81,13.1297222222222],[100.810277777778,13.1294444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1547222222222],[100.810555555556,13.155]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1413888888889],[100.81,13.1411111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.14],[100.81,13.1402777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1294444444444],[100.810277777778,13.1291666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1291666666667],[100.810277777778,13.1288888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1288888888889],[100.810277777778,13.1286111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1286111111111],[100.810277777778,13.1283333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810277777778,13.1283333333333],[100.810555555556,13.1280555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810555555556,13.1563888888889],[100.810833333333,13.1566666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810555555556,13.155],[100.810833333333,13.155]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810555555556,13.1416666666667],[100.810277777778,13.1413888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810555555556,13.14],[100.810277777778,13.14]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810555555556,13.1280555555556],[100.810833333333,13.1277777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1569444444444],[100.811111111111,13.1572222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1566666666667],[100.810833333333,13.1569444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1552777777778],[100.811111111111,13.1555555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.155],[100.810833333333,13.1552777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1419444444444],[100.810555555556,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1397222222222],[100.810555555556,13.14]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1277777777778],[100.810833333333,13.1275]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.810833333333,13.1275],[100.811111111111,13.1272222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811111111111,13.1572222222222],[100.811388888889,13.1575]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811111111111,13.1555555555556],[100.811388888889,13.1555555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811111111111,13.1397222222222],[100.810833333333,13.1397222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811388888889,13.1575],[100.811666666667,13.1577777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811388888889,13.1555555555556],[100.811666666667,13.1558333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811388888889,13.1488888888889],[100.811666666667,13.1488888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811666666667,13.1577777777778],[100.811944444444,13.1580555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811666666667,13.1558333333333],[100.811944444444,13.1561111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811666666667,13.1488888888889],[100.811944444444,13.1491666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811944444444,13.1561111111111],[100.812222222222,13.1563888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811944444444,13.1491666666667],[100.812222222222,13.1494444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811944444444,13.1358333333333],[100.811944444444,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811944444444,13.1355555555556],[100.811944444444,13.1352777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.811944444444,13.1352777777778],[100.812222222222,13.135]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1563888888889],[100.8125,13.1566666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1505555555556],[100.8125,13.1508333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1502777777778],[100.812222222222,13.1505555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.15],[100.812222222222,13.1502777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1497222222222],[100.812222222222,13.15]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1494444444444],[100.812222222222,13.1497222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1363888888889],[100.812222222222,13.1361111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.1361111111111],[100.811944444444,13.1358333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812222222222,13.135],[100.8125,13.135]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8125,13.1508333333333],[100.812777777778,13.1511111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8125,13.135],[100.812777777778,13.135]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812777777778,13.1511111111111],[100.813055555556,13.1513888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812777777778,13.1483333333333],[100.813055555556,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812777777778,13.135],[100.813055555556,13.135]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.812777777778,13.1327777777778],[100.813055555556,13.1327777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1552777777778],[100.813333333333,13.1552777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1513888888889],[100.813333333333,13.1516666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1486111111111],[100.813333333333,13.1488888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1441666666667],[100.813333333333,13.1438888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1355555555556],[100.813055555556,13.1352777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1352777777778],[100.813055555556,13.135]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.135],[100.813333333333,13.1347222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813055555556,13.1327777777778],[100.813333333333,13.1327777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1552777777778],[100.813611111111,13.1555555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1516666666667],[100.813611111111,13.1516666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1488888888889],[100.813611111111,13.1491666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1438888888889],[100.813611111111,13.1436111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1363888888889],[100.813333333333,13.1361111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1361111111111],[100.813333333333,13.1358333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1358333333333],[100.813055555556,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1347222222222],[100.813611111111,13.1344444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813333333333,13.1327777777778],[100.813611111111,13.1325]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813611111111,13.1516666666667],[100.813888888889,13.1516666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813611111111,13.1491666666667],[100.813888888889,13.1491666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813611111111,13.1436111111111],[100.813888888889,13.1436111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813611111111,13.1344444444444],[100.813888888889,13.1344444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813611111111,13.1325],[100.813888888889,13.1325]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813888888889,13.1516666666667],[100.814166666667,13.1519444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813888888889,13.1491666666667],[100.814166666667,13.1491666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813888888889,13.1436111111111],[100.814166666667,13.1436111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813888888889,13.1344444444444],[100.814166666667,13.1344444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.813888888889,13.1325],[100.814166666667,13.1325]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1519444444444],[100.814444444444,13.1522222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1491666666667],[100.814444444444,13.1488888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1436111111111],[100.814166666667,13.1433333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1433333333333],[100.814444444444,13.1430555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1344444444444],[100.814444444444,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814166666667,13.1325],[100.814444444444,13.1322222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814444444444,13.1522222222222],[100.814722222222,13.1525]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814444444444,13.1488888888889],[100.814722222222,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814444444444,13.1430555555556],[100.814722222222,13.1427777777778]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814444444444,13.1341666666667],[100.814722222222,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814444444444,13.1322222222222],[100.814722222222,13.1322222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814722222222,13.1486111111111],[100.815,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814722222222,13.1427777777778],[100.815,13.1425]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814722222222,13.1416666666667],[100.815,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814722222222,13.1341666666667],[100.815,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.814722222222,13.1322222222222],[100.815,13.1322222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815,13.1486111111111],[100.815277777778,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815,13.1425],[100.815277777778,13.1422222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815,13.1416666666667],[100.815277777778,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815,13.1341666666667],[100.815277777778,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815,13.1322222222222],[100.815277777778,13.1322222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815277777778,13.1483333333333],[100.815555555556,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815277777778,13.1422222222222],[100.815277777778,13.1419444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815277777778,13.1419444444444],[100.815555555556,13.1419444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815277777778,13.1416666666667],[100.815555555556,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815277777778,13.1341666666667],[100.815555555556,13.1344444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815555555556,13.1486111111111],[100.815833333333,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815555555556,13.1419444444444],[100.815833333333,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815555555556,13.1416666666667],[100.815833333333,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815555555556,13.1344444444444],[100.815833333333,13.1344444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815833333333,13.1486111111111],[100.816111111111,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815833333333,13.1416666666667],[100.816111111111,13.1413888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.815833333333,13.1344444444444],[100.816111111111,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816111111111,13.1486111111111],[100.816388888889,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816111111111,13.1477777777778],[100.816388888889,13.1480555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816111111111,13.1413888888889],[100.816388888889,13.1416666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816111111111,13.1366666666667],[100.816388888889,13.1363888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816111111111,13.1341666666667],[100.816388888889,13.1341666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816388888889,13.1486111111111],[100.816666666667,13.1486111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816388888889,13.1480555555556],[100.816666666667,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816388888889,13.1416666666667],[100.816666666667,13.1419444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816388888889,13.1363888888889],[100.816666666667,13.1361111111111]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816388888889,13.1341666666667],[100.816666666667,13.1338888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816666666667,13.1486111111111],[100.816944444444,13.1488888888889]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816666666667,13.1483333333333],[100.816944444444,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816666666667,13.1419444444444],[100.816944444444,13.1422222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816666666667,13.1361111111111],[100.816944444444,13.1358333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816944444444,13.1488888888889],[100.817222222222,13.1491666666667]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816944444444,13.1483333333333],[100.817222222222,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816944444444,13.1422222222222],[100.817222222222,13.1422222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.816944444444,13.1358333333333],[100.817222222222,13.1358333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1497222222222],[100.817222222222,13.15]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1494444444444],[100.817222222222,13.1497222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1491666666667],[100.817222222222,13.1494444444444]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1483333333333],[100.8175,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1422222222222],[100.8175,13.1422222222222]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.817222222222,13.1358333333333],[100.8175,13.1355555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8175,13.1483333333333],[100.817777777778,13.1483333333333]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.8175,13.1422222222222],[100.817777777778,13.1425]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.818333333333,13.1383333333333],[100.818611111111,13.1380555555556]]}},{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[100.818611111111,13.1380555555556],[100.818888888889,13.1377777777778]]}}]},"stations":{"type":"FeatureCollection","features":[{"type":"Feature","properties":{"site_id":"B01"},"geometry":{"type":"Point","coordinates":[100.806666666667,13.1422222222222]}},{"type":"Feature","properties":{"site_id":"B02"},"geometry":{"type":"Point","coordinates":[100.809722222222,13.1605555555556]}},{"type":"Feature","properties":{"site_id":"B03"},"geometry":{"type":"Point","coordinates":[100.816666666667,13.1338888888889]}},{"type":"Feature","properties":{"site_id":"B04"},"geometry":{"type":"Point","coordinates":[100.808611111111,13.1619444444444]}},{"type":"Feature","properties":{"site_id":"B05"},"geometry":{"type":"Point","coordinates":[100.817777777778,13.1425]}},{"type":"Feature","properties":{"site_id":"B06"},"geometry":{"type":"Point","coordinates":[100.807777777778,13.1652777777778]}},{"type":"Feature","properties":{"site_id":"B07"},"geometry":{"type":"Point","coordinates":[100.811111111111,13.1272222222222]}},{"type":"Feature","properties":{"site_id":"B08"},"geometry":{"type":"Point","coordinates":[100.8125,13.1566666666667]}},{"type":"Feature","properties":{"site_id":"B09"},"geometry":{"type":"Point","coordinates":[100.814722222222,13.1525]}},{"type":"Feature","properties":{"site_id":"B10"},"geometry":{"type":"Point","coordinates":[100.806944444444,13.1458333333333]}},{"type":"Feature","properties":{"site_id":"B11"},"geometry":{"type":"Point","coordinates":[100.817222222222,13.15]}},{"type":"Feature","properties":{"site_id":"B12"},"geometry":{"type":"Point","coordinates":[100.817777777778,13.1483333333333]}}]},"coastline":{"type":"Polygon","coordinates":[[[100.7983177,13.1634321],[100.7983619,13.1633023],[100.7984423,13.1632405],[100.7985088,13.1632085],[100.7986699,13.1631402],[100.7989083,13.1630842],[100.7992476,13.1630583],[100.7994993,13.1631283],[100.7997598,13.1630504],[100.8000846,13.1630844],[100.8003547,13.1631361],[100.8006276,13.1629859],[100.8008087,13.1627622],[100.8008803,13.1627055],[100.8009187,13.1625819],[100.8010509,13.1624349],[100.8011409,13.1623329],[100.8011397,13.1621313],[100.8013562,13.1614481],[100.8014258,13.1609144],[100.8015074,13.1604109],[100.8013037,13.1598277],[100.8013282,13.1596597],[100.8014725,13.1594255],[100.8015369,13.1587299],[100.8017168,13.1585431],[100.8027742,13.1575966],[100.802848,13.1571913],[100.8030913,13.1570004],[100.8031883,13.1567516],[100.803616,13.1562866],[100.8039394,13.1562577],[100.8043433,13.155818],[100.8047127,13.1554137],[100.8050341,13.1551678],[100.8050954,13.1549553],[100.8051118,13.1548097],[100.8050503,13.1547124],[100.8050081,13.1546356],[100.8049446,13.1546096],[100.8048952,13.1545464],[100.804837,13.1544849],[100.8048292,13.1544313],[100.8048503,13.1543596],[100.8047874,13.1542608],[100.8047202,13.1542597],[100.8047101,13.1542919],[100.8046689,13.1543016],[100.8046388,13.1542643],[100.8046203,13.154227],[100.8045551,13.1541442],[100.8044345,13.1540197],[100.8042214,13.1538275],[100.8039635,13.1534495],[100.8038019,13.1533502],[100.8036538,13.1533339],[100.8035041,13.1531916],[100.803631,13.1529063],[100.8037075,13.15268],[100.803702,13.1523032],[100.8038658,13.1515896],[100.803988,13.1508894],[100.8041244,13.1501953],[100.804195,13.149737],[100.8040916,13.1494773],[100.8038199,13.149266],[100.8031008,13.1488483],[100.8023426,13.1487642],[100.8021152,13.1489344],[100.8018735,13.1489903],[100.8011317,13.1489713],[100.8007725,13.1487343],[100.8005358,13.1486082],[100.7997363,13.1488292],[100.7994325,13.1485298],[100.7993049,13.148041],[100.7994928,13.1475792],[100.800144,13.1474212],[100.8004434,13.1473096],[100.800843,13.1470825],[100.8012677,13.1472486],[100.8014955,13.1472447],[100.8018081,13.1475337],[100.802052,13.1474857],[100.8021692,13.1475296],[100.8023148,13.1477584],[100.8025433,13.1478454],[100.8028701,13.1480247],[100.8031895,13.1480185],[100.8036821,13.1479098],[100.8040928,13.1477372],[100.8042867,13.1475031],[100.8044592,13.1475379],[100.8050898,13.1473003],[100.8055305,13.1471004],[100.8059432,13.1465735],[100.8062002,13.1462679],[100.8064469,13.1461524],[100.8070641,13.1450713],[100.8071671,13.1446464],[100.8072395,13.144437],[100.8072763,13.1442851],[100.8072677,13.1437234],[100.8073008,13.1436015],[100.8067751,13.1424277],[100.8061638,13.1418064],[100.8060238,13.1413101],[100.8055212,13.1409193],[100.8053159,13.1406555],[100.8045911,13.1406454],[100.8041796,13.1401804],[100.8041261,13.1400588],[100.8037137,13.1393318],[100.8034538,13.1388422],[100.8034717,13.1385351],[100.8035669,13.1383092],[100.8037572,13.138089],[100.8044329,13.1373488],[100.8048849,13.136824],[100.8049887,13.1364611],[100.8052386,13.1361193],[100.8059711,13.1359078],[100.8061529,13.1357896],[100.8062665,13.1357791],[100.8063168,13.1356042],[100.8065789,13.1355714],[100.8066805,13.1354244],[100.8066809,13.1352134],[100.8069869,13.1350546],[100.8071144,13.1349185],[100.8071273,13.1345802],[100.8071516,13.134094],[100.807416,13.1336719],[100.8075667,13.1335796],[100.8076693,13.1332672],[100.8077222,13.1327793],[100.8075647,13.1323397],[100.8075252,13.1317388],[100.8075616,13.130728],[100.8077519,13.1303905],[100.8078129,13.1301421],[100.807771,13.1298566],[100.8078319,13.1295785],[100.8075996,13.1292077],[100.8075692,13.1289667],[100.8076377,13.1287924],[100.8076149,13.128607],[100.8073497,13.1283571],[100.8075035,13.1278858],[100.8072947,13.1274196],[100.8075528,13.1264987],[100.8077436,13.1260318],[100.8080107,13.1257298],[100.8082188,13.1256086],[100.8085287,13.1257297],[100.8088789,13.1254998],[100.8090655,13.1254924],[100.8092119,13.1252999],[100.8094462,13.125281],[100.8097039,13.1253783],[100.8099044,13.12545],[100.8100916,13.1257276],[100.8102071,13.1259622],[100.8102203,13.1263546],[100.81044,13.1265653],[100.8105342,13.1269802],[100.8105819,13.1270437],[100.81068,13.1271326],[100.8107076,13.1271595],[100.8110005,13.1269103],[100.8111291,13.126948],[100.811191,13.1271308],[100.8113892,13.1273586],[100.811436,13.1274924],[100.81163,13.1277715],[100.8118272,13.1279533],[100.8120806,13.1284268],[100.8121783,13.1284601],[100.8122748,13.1286707],[100.8126267,13.1289351],[100.8131953,13.1292397],[100.8136901,13.1295245],[100.8140838,13.1299222],[100.8143164,13.130501],[100.81487,13.1310159],[100.8154253,13.1312721],[100.8156164,13.1315342],[100.8156039,13.1317253],[100.8154908,13.131878],[100.815498,13.1320758],[100.8154703,13.132102],[100.8154951,13.1322421],[100.8155714,13.132476],[100.8157779,13.1327281],[100.8159894,13.1328744],[100.8161074,13.132808],[100.816203,13.1328592],[100.8161446,13.1330798],[100.8166865,13.1334709],[100.8168675,13.1334246],[100.8169762,13.133315],[100.8170953,13.1332705],[100.8171906,13.1333402],[100.8171084,13.1334118],[100.8171691,13.1335223],[100.8170558,13.1335816],[100.816961,13.133555],[100.8168233,13.1336629],[100.8167931,13.133846],[100.8167238,13.1340504],[100.8166942,13.1343917],[100.8168543,13.1346338],[100.8171765,13.1348993],[100.8172866,13.1351782],[100.8174697,13.1353352],[100.8176356,13.1356327],[100.8182365,13.1360266],[100.8184863,13.1363163],[100.8188454,13.1365744],[100.818992,13.1367624],[100.8188373,13.1372317],[100.8190098,13.1376198],[100.8191526,13.1377067],[100.8194144,13.1377994],[100.8195274,13.1379269],[100.8198176,13.1386217],[100.8199082,13.1390102],[100.8198907,13.1394074],[100.8196999,13.1393631],[100.8195158,13.1394882],[100.8194917,13.1396058],[100.8195043,13.1398117],[100.8194815,13.1398532],[100.8192716,13.1398097],[100.8191763,13.139867],[100.8189436,13.140988],[100.8189617,13.1411511],[100.8190281,13.1413656],[100.8192029,13.1414744],[100.8197227,13.1416786],[100.8198903,13.1416492],[100.8215518,13.1413289],[100.8219868,13.1412507],[100.8221095,13.1412051],[100.8223058,13.1411138],[100.8223928,13.1411073],[100.8224843,13.1411682],[100.822511,13.1412485],[100.8224664,13.1413311],[100.8224084,13.1413658],[100.8222746,13.1413854],[100.821882,13.1414592],[100.821418,13.1415635],[100.8209075,13.1416711],[100.8207887,13.1416962],[100.8196357,13.1419393],[100.8196198,13.1419856],[100.819678,13.1420331],[100.8196938,13.1422174],[100.819637,13.1422297],[100.8195092,13.1421913],[100.8192632,13.1420976],[100.8190669,13.1420411],[100.8189979,13.1420575],[100.8187481,13.1420981],[100.8181591,13.1422082],[100.8180577,13.1423439],[100.8180201,13.142549],[100.8180448,13.1425589],[100.8181243,13.1427056],[100.81812,13.1428107],[100.8179885,13.1428002],[100.8178783,13.1427098],[100.8176996,13.1427928],[100.8175726,13.1429387],[100.8174881,13.1431367],[100.8174534,13.1434049],[100.8174679,13.1435744],[100.8175785,13.1435922],[100.8176597,13.1435592],[100.8177451,13.1436387],[100.8178398,13.1437045],[100.8180223,13.1442764],[100.818334,13.1453366],[100.8183137,13.1456192],[100.818288,13.1458712],[100.8183587,13.1462605],[100.8184645,13.1465443],[100.8184931,13.1468506],[100.8183805,13.1472098],[100.8182778,13.1474148],[100.8181953,13.1475496],[100.8179931,13.1478572],[100.8179715,13.1480071],[100.81797,13.1481305],[100.8179863,13.1483146],[100.8179831,13.1484921],[100.818054,13.1486394],[100.8182734,13.148933],[100.8185525,13.149221],[100.81876,13.1493512],[100.8194316,13.1497591],[100.8198718,13.149996],[100.8200736,13.1499994],[100.8201806,13.150124],[100.8201664,13.1502056],[100.820045,13.1501935],[100.8199617,13.1501685],[100.8198594,13.1502009],[100.8197237,13.1503939],[100.8197191,13.1504998],[100.8197545,13.1505787],[100.8197238,13.1506064],[100.8196072,13.1505763],[100.8194359,13.1506296],[100.8192823,13.1506071],[100.8190468,13.1504252],[100.8186707,13.1501333],[100.8181607,13.1499196],[100.8180012,13.1499275],[100.8179586,13.1499334],[100.8178885,13.1499379],[100.8174749,13.1500294],[100.8169641,13.1502332],[100.8166453,13.1504383],[100.8160282,13.1509641],[100.815387,13.1514503],[100.8152789,13.151754],[100.8151684,13.1519407],[100.8151907,13.1520278],[100.8150707,13.152344],[100.8149307,13.1526947],[100.8147055,13.1530269],[100.8146552,13.1532131],[100.8144097,13.1534831],[100.8141362,13.1539235],[100.8142532,13.1540328],[100.814136,13.1542843],[100.8141563,13.1545456],[100.8138915,13.1549069],[100.8139187,13.1552075],[100.8137899,13.1553153],[100.8136028,13.1556302],[100.8131248,13.1564028],[100.8130167,13.1566398],[100.8129101,13.1567059],[100.8127377,13.1567562],[100.8125228,13.1570166],[100.812503,13.1572914],[100.8124716,13.157585],[100.8125408,13.1576642],[100.8121215,13.1580219],[100.8122951,13.1581157],[100.8122315,13.1585059],[100.8121954,13.158697],[100.8122491,13.1588309],[100.8123703,13.1590853],[100.8125046,13.1592837],[100.8125037,13.1594087],[100.8124377,13.1594829],[100.8123757,13.1595298],[100.8121513,13.1595412],[100.8121676,13.1596821],[100.8120597,13.1599327],[100.8119546,13.1601971],[100.8119526,13.160282],[100.8118949,13.1603281],[100.8115536,13.1604563],[100.8115237,13.1604675],[100.8114815,13.1604834],[100.8111751,13.1605985],[100.8110786,13.160576],[100.8109843,13.160513],[100.8108896,13.1602242],[100.8107182,13.1602845],[100.81042,13.1604411],[100.8100508,13.1606491],[100.8100001,13.1606817],[100.8099565,13.1607098],[100.809918,13.1607346],[100.8095839,13.1609497],[100.8093989,13.1609993],[100.8092233,13.1610464],[100.8091499,13.1610713],[100.8090348,13.1614541],[100.8089625,13.1617107],[100.8089515,13.1617494],[100.8088962,13.1619457],[100.8087957,13.1625527],[100.8088219,13.1626625],[100.8088685,13.1628527],[100.8086802,13.16348],[100.8083543,13.1636772],[100.808117,13.1642564],[100.807819,13.1648413],[100.8078374,13.1655482],[100.8076434,13.1659315],[100.8076516,13.1660616],[100.8076606,13.1662057],[100.8076644,13.1662661],[100.8075171,13.166831],[100.8075137,13.1673784],[100.8078672,13.1679638],[100.808094,13.1686383],[100.8086315,13.1693557],[100.8087246,13.1694989],[100.808905,13.1700564],[100.8089171,13.1703941],[100.8091204,13.1707407],[100.8096071,13.1716178],[100.8096598,13.1723989],[100.8093485,13.1730671],[100.8088785,13.1734784],[100.808474,13.1736811],[100.8080159,13.1740287],[100.8077363,13.1744921],[100.8076827,13.1747411],[100.8075816,13.1750945],[100.8075519,13.1754826],[100.8074129,13.1760101],[100.8078553,13.1766353],[100.8080932,13.177452],[100.8083252,13.1780834],[100.8083728,13.1787437],[100.8082717,13.1792767],[100.8080833,13.1795272],[100.8076232,13.1799602],[100.8072438,13.1800684],[100.8067487,13.1799775],[100.8067542,13.1799742],[100.8061561,13.1797813],[100.8060571,13.1793821],[100.8058993,13.1788669],[100.8056124,13.1786163],[100.8051518,13.1785526],[100.8051365,13.1780892],[100.8046343,13.1779221],[100.8039863,13.1778914],[100.8038573,13.1779378],[100.8034946,13.1784136],[100.8030722,13.1789928],[100.8025836,13.1792527],[100.8023457,13.1792791],[100.8020684,13.1793018],[100.8016071,13.1790501],[100.8013482,13.178697],[100.8008457,13.1780123],[100.8004051,13.1771643],[100.8004798,13.1762387],[100.8006982,13.1753293],[100.8010114,13.1747179],[100.8012658,13.1741379],[100.8014479,13.1738079],[100.8016247,13.1733338],[100.8016729,13.1729462],[100.8017895,13.1722532],[100.8017876,13.1717322],[100.8017556,13.1713864],[100.8013901,13.1710342],[100.800104,13.1704481],[100.7996866,13.1700296],[100.7997105,13.1696941],[100.7990169,13.1689004],[100.7989142,13.1685775],[100.7991105,13.1680629],[100.799609,13.1665932],[100.7995797,13.1662168],[100.7997114,13.165931],[100.7998835,13.1652616],[100.7997044,13.1648753],[100.7993519,13.1644886],[100.7983177,13.1634321]]]},"label_positions":{"B01":[100.81103219696966,13.140410353535355],"B02":[100.80612058080806,13.15936710858586],"B03":[100.81324324324324,13.135921755088422],"B04":[100.80401858304297,13.163796941540843],"B05":[100.81400097465887,13.1431432748538],"B06":[100.80452252944188,13.167154377880182],"B07":[100.809974503489,13.130597155126143],"B08":[100.80917112299464,13.153223410576352],"B09":[100.81176523297493,13.150409199522104],"B10":[100.80946151458721,13.147053072625699],"B11":[100.81340632603404,13.14847729115977],"B12":[100.81483950617285,13.147129629629635]},"status":"B01\u2013B12: original R/D8 catchments, modeled outlets and runoff paths \u00b7 AWS/Mapzen Skadi ~30 m + OSM boundary \u00b7 30 Sep 2026"}''')
INITIAL_GIS = BUILTIN_GIS.copy()
_extra_paths = {k: GIS_DIR/v for k, v in GIS_FILES.items() if (GIS_DIR/v).is_file()}
if _extra_paths:
    _extra = load_gis_bundle(_extra_paths)
    INITIAL_GIS.update({k:v for k,v in _extra.items() if k != 'status'})
    INITIAL_GIS['status'] += '; local overrides: ' + _extra['status']



def gis_status(bundle, raster='None'):
    text = html.escape((bundle or {}).get('status', 'No GIS files loaded. Demonstration coordinates only.'))
    legend = ''
    if raster == RASTER_NAMES[1]:
        legend = '<br><b>Slope (degrees):</b> pale green 0° → green 15° → dark teal 30° → brown ≥45°.'
    elif raster == RASTER_NAMES[2]:
        legend = '<br><b>Slope-only screening:</b> dark green &lt;5° · light green 5–15° · sand &gt;15°. UNVALIDATED: gentle slopes do not prove groundwater; no aquifer, yield, well or recharge measurements.'
    key = 'slope' if raster == RASTER_NAMES[1] else 'groundwater' if raster == RASTER_NAMES[2] else None
    if key and key not in (bundle or {}):
        legend += '<br><b>Selected raster is unavailable. Upload its GeoTIFF to display it.</b>'
    return f'<div class="ow-banner"><b>GIS / DATA STATUS</b><br>{text}{legend}<br>พื้นที่และจุดระบายคำนวณจากภูมิประเทศเดิม ไม่ใช่สถานีเก็บตัวอย่างที่สำรวจยืนยัน · สีพื้นที่แยก B01–B12; จุดแสดงคะแนนจำลอง · ค่าน้ำและคะแนนทั้งหมดเป็นข้อมูลจำลอง ไม่ใช่ผลตรวจพื้นที่จริง · © OpenStreetMap contributors / AWS–Mapzen Skadi</div>'


def map_plot(bundle=None, layers=None, raster='None', show_risk=False):
    bundle = INITIAL_GIS if bundle is None else bundle
    layers = LAYER_NAMES if layers is None else layers
    fig = go.Figure()
    coast = bundle.get('coastline', BUILTIN_GIS['coastline'])
    ring = coast['coordinates'][0]
    fig.add_trace(go.Scattermap(lon=[p[0] for p in ring],lat=[p[1] for p in ring],
        mode='lines',line=dict(color='#acb9b7',width=1),
        hoverinfo='skip',showlegend=False,name='Ko Sichang coastline'))
    if 'Drainage catchments' in layers and 'catchments' in bundle:
        feats = bundle['catchments']['features']
        ids = [f['properties']['site_id'] for f in feats]
        values = [float(LATEST.loc[s].risk_score) for s in ids] if show_risk else [IDS.index(s) for s in ids]
        fig.add_trace(go.Choroplethmap(
            geojson=bundle['catchments'], locations=ids, featureidkey='properties.site_id', z=values,
            zmin=0, zmax=100 if show_risk else 12,
            colorscale=[[0,'#16877e'],[.5,'#edc873'],[1,'#bf5651']] if show_risk else [[0.0, '#fbb9b9'], [0.08333333333333333, '#fbb9b9'], [0.08333333333333333, '#efbd96'], [0.16666666666666666, '#efbd96'], [0.16666666666666666, '#d8c777'], [0.25, '#d8c777'], [0.25, '#b9d182'], [0.3333333333333333, '#b9d182'], [0.3333333333333333, '#7fd49c'], [0.4166666666666667, '#7fd49c'], [0.4166666666666667, '#7dd9b4'], [0.5, '#7dd9b4'], [0.5, '#6cd1d3'], [0.5833333333333334, '#6cd1d3'], [0.5833333333333334, '#7bcce9'], [0.6666666666666666, '#7bcce9'], [0.6666666666666666, '#a0baf5'], [0.75, '#a0baf5'], [0.75, '#c9adf2'], [0.8333333333333334, '#c9adf2'], [0.8333333333333334, '#e4a8eb'], [0.9166666666666666, '#e4a8eb'], [0.9166666666666666, '#eeadd7'], [1.0, '#eeadd7']],
            marker_opacity=.75, marker_line_width=1.3, marker_line_color='#366a70',
            showscale=bool(show_risk), colorbar=dict(title='Demo risk', thickness=10, len=.6),
            customdata=ids, name='B01–B12 · modeled drainage basins',
            hovertemplate='%{customdata}<br>ลุ่มรับน้ำจากแบบจำลองภูมิประเทศ' + ('<br>Synthetic score %{z:.1f} / 100 (illustrative B-ID join)' if show_risk else '') + '<extra></extra>'))
    if 'Drainage catchments' in layers and 'catchments' in bundle:
        from shapely.geometry import shape
        labels = []
        for feature in bundle['catchments']['features']:
            p = shape(feature['geometry']).centroid
            labels.append((feature['properties']['site_id'], p.x, p.y))
        fig.add_trace(go.Scattermap(lon=[p[1] for p in labels],lat=[p[2] for p in labels],
            mode='text',text=[p[0] for p in labels],textfont=dict(size=12,color='#153b4a'),
            hoverinfo='skip',showlegend=False,name='Basin labels'))
    if 'Flow paths' in layers and 'flows' in bundle:
        xs, ys = [], []
        for f in bundle['flows']['features']:
            g = f['geometry']
            for line in ([g['coordinates']] if g['type']=='LineString' else g['coordinates']):
                xs.extend([p[0] for p in line] + [None]); ys.extend([p[1] for p in line] + [None])
        fig.add_trace(go.Scattermap(lon=xs, lat=ys, mode='lines', line=dict(color='#2c91ba',width=2),
                                   name='Modeled flow paths', hoverinfo='name'))
    if 'Monitoring points' in layers:
        if 'stations' in bundle:
            rows = [(f['properties']['site_id'], *f['geometry']['coordinates'][:2]) for f in bundle['stations']['features']]
            d = pd.DataFrame(rows, columns=['site_id','longitude','latitude']).merge(LATEST.reset_index().drop(columns=['longitude','latitude']),on='site_id')
            point_label = 'จุดระบายจาก GIS · คะแนนจำลอง (ไม่ใช่สถานีตรวจยืนยัน)'
        else:
            d = LATEST.reset_index()
            point_label = 'SIMULATED coordinates · synthetic values'
        fig.add_trace(go.Scattermap(lon=d.longitude,lat=d.latitude,mode='markers',
            marker=dict(size=12,color=d.risk_score,cmin=0,cmax=100,colorscale=[[0,'#16877e'],[.5,'#edc873'],[1,'#bf5651']]),
            customdata=np.column_stack([d.site_id,d.risk_score]),name=point_label,
            hovertemplate='%{customdata[0]}<br>'+point_label+'<br>Demo risk %{customdata[1]:.1f}/100<extra></extra>'))
    if not fig.data:
        fig.add_trace(go.Scattermap(lon=[100.811], lat=[13.153], marker=dict(size=0), hoverinfo='skip',showlegend=False))
    overlays = []
    key = 'slope' if raster == RASTER_NAMES[1] else 'groundwater' if raster == RASTER_NAMES[2] else None
    if key in bundle:
        overlays.append(dict(bundle[key], below='traces'))
    fig.update_layout(map=dict(style='https://basemaps.cartocdn.com/gl/positron-nolabels-gl-style/style.json',center=dict(lat=13.1525,lon=100.810),zoom=12.5,layers=overlays),
                      showlegend=False, uirevision='ko-sichang-map')
    return styled(fig, 'เกาะสีชัง · 12 ลุ่มรับน้ำและจุดระบาย B01–B12', 640)


def upload_gis(catchments, flows, stations, slope, groundwater, layers, raster, show_risk):
    overrides = load_gis_bundle(dict(zip(GIS_FILES, [catchments,flows,stations,slope,groundwater])))
    bundle = BUILTIN_GIS.copy()
    bundle.update({k:v for k,v in overrides.items() if k != 'status'})
    bundle['status'] += '; uploaded overrides: ' + overrides['status']
    return bundle, map_plot(bundle,layers,raster,show_risk), gis_status(bundle,raster)


def refresh_map(bundle, layers, raster, show_risk):
    return map_plot(bundle,layers,raster,show_risk), gis_status(bundle,raster)


# Shared presentation helpers; numerical/model functions above retain original behavior.
def kpi(label, value, foot='SYNTHETIC DATA'):
    return f'<div class="ow-kpi"><div class="ow-kpi-label">{html.escape(label)}</div><div class="ow-kpi-value">{html.escape(str(value))}</div><div class="ow-kpi-foot">{html.escape(foot)}</div></div>'


def section(title, subtitle='', badge=''):
    return f'<div class="ow-section"><div><h2>{html.escape(title)}</h2><p>{html.escape(subtitle)}</p></div><span class="ow-badge">{html.escape(badge)}</span></div>'


def result_card(site, model, original, score, delta):
    color = '#16877e' if score < 35 else '#c29539' if score < 65 else '#bc5651'
    return f'''<div class="ow-section"><div><h2>2. ผลประเมินระดับความเสี่ยงจากโมเดลที่ฝึกแล้ว</h2><p>{html.escape(model)}</p></div><span class="ow-badge">{site}</span></div>
    <div class="ow-score"><div class="ow-ring" role="img" aria-label="Synthetic risk score {score:.1f} out of 100" style="background:conic-gradient({color} {score}%,#ecf1f0 {score}%);"><div class="ow-ring-center"><div class="ow-ring-score" style="color:{color}">{score:.1f}</div><div class="ow-ring-label">คะแนน / 100 · สาธิต</div></div></div>
    <div class="ow-score-text"><h3 style="color:{color}">{category(score)}</h3><p>คะแนนลำดับตรวจสอบสาธิต<br>ไม่ใช่โอกาสเกิดโรคจริง<br>ต่ำ &lt;35 · ปานกลาง 35–&lt;65 · สูง ≥65</p></div></div>
    <div class="ow-action"><strong>{action(score)}</strong>ตรวจความครบถ้วนของข้อมูลและแหล่งที่มา<br>ส่งข้อมูลให้ผู้รับผิดชอบพื้นที่กำหนดการติดตามที่เหมาะสม</div>
    <div class="ow-mini-grid"><div><small>คะแนนอ้างอิง (สูตรจำลอง)</small><b>{original:.1f}</b></div><div><small>ผลต่างเทียบคะแนนอ้างอิง</small><b>{delta:+.1f}</b></div><div><small>สถานะข้อมูล</small><b style="font-size:13px">SYNTHETIC ONLY</b></div></div>'''


def simulator_view(site, model, *values):
    original, score, delta, cat, fig, note = simulate(site,model,*values)
    return result_card(site,model,original,score,delta), fig, note


def reset_simulator(site, model):
    vals = load_sim_site(site)
    return (*vals, *simulator_assess(site,model,*vals[:-1]))


def input_fingerprint(site, model, values):
    return [site, model, *[round(float(v), 2) if v is not None else None for v in values]]


def simulator_assess(site, model, *values):
    return (*simulator_view(site, model, *values), input_fingerprint(site, model, values))


def pending_result(evaluated, site, model, *values):
    if evaluated == input_fingerprint(site, model, values):
        return gr.skip()
    return '<div class="ow-banner"><b>Inputs changed / ข้อมูลเปลี่ยนแล้ว</b><br>กด “ประเมินความเสี่ยง” เพื่ออัปเดตผลด้วยค่าที่แก้ไข</div>'


def performance_plot():
    fig = go.Figure()
    for metric, color in [('MAE','#16877e'),('RMSE','#648391')]:
        fig.add_trace(go.Bar(x=['Ridge','MLP'], y=METRICS[metric],name=metric,marker_color=color))
    fig.update_layout(barmode='group',legend=dict(x=0,y=1.15))
    fig.update_yaxes(title='Score error (lower is better)',rangemode='tozero')
    return styled(fig,'Held-out daily prediction · synthetic data',290)


CSS = ":root {color-scheme:light!important}\nbody,.gradio-container {background:#f4f7f7!important;color:#193b48!important}\n.gradio-container {max-width:1480px!important;margin:auto!important;padding:0 28px 36px!important;font-family:'Noto Sans Thai',Tahoma,Arial,sans-serif!important}\n.ow-header {background:#152f3a;color:white;margin:0 -28px 28px;padding:25px 34px;display:flex;justify-content:space-between;align-items:center;gap:20px}\n.ow-brand {font-size:31px;font-weight:800;letter-spacing:-1px;line-height:1.2}.ow-brand span{color:#87cdbc}.ow-header small{display:block;color:#b2cbd0;letter-spacing:2.2px;font-size:10px;margin-top:8px}.ow-header .ow-badge{border:1px solid #6a9999;color:#d6ebe6;background:#284c54}\n.ow-badge{display:inline-block;background:#eaf5f1;color:#16877e;border-radius:7px;padding:9px 13px;font-size:11px;letter-spacing:.5px}\n.ow-eyebrow{color:#16877e;font-size:11px;font-weight:800;letter-spacing:2px;margin:4px 0 9px}.ow-title{font-size:29px;font-weight:800;line-height:1.4;margin:0;color:#193b48}.ow-subtitle{font-size:14px;color:#6b8590;margin:9px 0 22px;line-height:1.7}\n.ow-banner {background:#fff8e8;border:1px solid #efd99d;color:#88672c;padding:14px 18px;border-radius:10px;font-size:12px;line-height:1.8;margin:2px 0 18px}\n.ow-panel {background:white!important;border:1px solid #d7e4e5!important;border-radius:13px!important;padding:20px!important;box-shadow:0 2px 4px #17374303!important;gap:15px!important;overflow:hidden}\n.ow-kpi{background:#fff;border:1px solid #d7e4e5;border-radius:12px;padding:19px 22px;min-height:112px}.ow-kpi-label{color:#6d858f;font-size:12px}.ow-kpi-value{font-size:30px;font-weight:800;color:#193b48;margin:6px 0}.ow-kpi-foot{font-size:11px;color:#16877e}\n.ow-section {display:flex;align-items:center;justify-content:space-between;gap:12px;border-bottom:1px solid #e7eeee;padding-bottom:15px;margin-bottom:2px}.ow-section h2{font-size:19px;margin:0;font-weight:800;color:#193b48}.ow-section p{font-size:12px;color:#6b8590;margin:6px 0 0}\n.ow-flow {display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin:5px 0 22px}.ow-flow>div{background:white;border:1px solid #d7e4e5;border-radius:10px;padding:14px 17px}.ow-flow small{display:block;color:#6b8590;font-size:10px;margin-bottom:8px}.ow-flow strong{font-size:13px;color:#193b48}\n.ow-score{display:flex;align-items:center;gap:25px;flex-wrap:wrap;margin:6px 0 20px}.ow-ring{width:170px;height:170px;border-radius:50%;display:flex;align-items:center;justify-content:center;flex:none}.ow-ring-center{width:147px;height:147px;border-radius:50%;background:white;display:flex;align-items:center;justify-content:center;flex-direction:column}.ow-ring-score{font-size:43px;font-weight:800;line-height:1.3}.ow-ring-label{font-size:11px;color:#6b8590}.ow-score-text h3{margin:0 0 10px;font-size:24px}.ow-score-text p{color:#6b8590;font-size:12px;line-height:1.8;margin:0}.ow-action{padding:19px;background:#eff7f4;border:1px solid #d2e8df;border-radius:10px;color:#193b48;font-size:13px;line-height:1.8}.ow-action strong{display:block;margin-bottom:7px}\n.ow-mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin:14px 0}.ow-mini-grid>div{padding:12px;background:#f6f9f9;border-radius:8px}.ow-mini-grid small{display:block;color:#6b8590;font-size:10px}.ow-mini-grid b{font-size:21px;color:#193b48}\n.ow-footer{margin-top:25px;border-top:1px solid #d7e4e5;padding-top:18px;color:#6b8590;font-size:11px;display:flex;justify-content:space-between;gap:15px}\n#navigation>.tab-nav{background:white;border:1px solid #d7e4e5!important;border-radius:10px;padding:6px!important;margin-bottom:23px;gap:7px}\n#navigation>.tab-nav button{padding:11px 20px!important;font-weight:700!important;color:#6b8590!important;border:0!important;border-radius:7px!important;font-size:13px!important}\n#navigation>.tab-nav button.selected{background:#e8f3f0!important;color:#127d72!important}\n.tabitem{padding:0!important;border:0!important;background:transparent!important;gap:20px!important}\n.gradio-container button.primary{background:#16877e!important;border:1px solid #16877e!important;color:white!important;border-radius:8px!important;font-weight:700!important;min-height:45px}\n.gradio-container .prose h3{color:#193b48!important;font-size:15px!important}.gradio-container .prose p,.gradio-container .prose li{line-height:1.8!important;font-size:12px}\n.gradio-container input,.gradio-container textarea{background:#f9fbfb!important;color:#193b48!important}.gradio-container label span{font-size:12px!important}.gradio-container .plot-container{border-radius:10px;overflow:hidden}\n.ow-task{background:#f9fbfb!important;border:1px solid #e0e9e9!important;border-radius:10px!important;padding:15px!important}\n@media(max-width:800px){.gradio-container{padding:0 14px 25px!important}.ow-header{margin:0 -14px 20px;padding:20px}.ow-header>.ow-badge{display:none}.ow-title{font-size:23px}.ow-flow{grid-template-columns:repeat(2,1fr)}.ow-flow>div:last-child{grid-column:1/-1}.ow-panel{padding:14px!important}.ow-score{gap:16px}.ow-ring{width:145px;height:145px}.ow-ring-center{width:125px;height:125px}#navigation>.tab-nav button{padding:10px!important;font-size:12px!important}.ow-mini-grid{grid-template-columns:1fr}.ow-kpi{padding:15px;min-height:100px}.ow-footer{flex-direction:column}}\n/* Explicit sizing prevents Gradio's intrinsic plot widths from widening mobile pages. */\n.gradio-container{width:100%!important;min-width:0!important;box-sizing:border-box!important}\n.gradio-container .app{padding:0!important;width:100%!important;min-width:0!important;box-sizing:border-box!important}\n.gradio-container .contain,.gradio-container .wrap,.gradio-container .tabs,.gradio-container .tab-wrapper,.gradio-container .tab-container,.gradio-container .tabitem{min-width:0!important;max-width:100%!important}\n.ow-header .ow-brand{color:#fff!important}.ow-header .ow-brand span{color:#87cdbc!important}\n#navigation .tab-container{display:flex;flex-wrap:wrap;background:white;border:1px solid #d7e4e5;border-radius:10px;padding:6px;margin-bottom:23px;gap:5px}\n#navigation .tab-container button{padding:11px 17px!important;border:0!important;border-radius:7px!important;color:#6b8590!important;font-size:13px!important;font-weight:700!important}\n#navigation .tab-container button.selected{background:#e8f3f0!important;color:#127d72!important}\n.gradio-container .html-container{padding:0!important}\n.gradio-container{--block-label-background-fill:transparent!important;--block-label-text-color:#54717c!important}\n.gradio-container .js-plotly-plot,.gradio-container .plot-container,.gradio-container .plotly{width:100%!important;max-width:100%!important;min-width:0!important}\n@media(max-width:800px){.gradio-container .row{flex-direction:column!important;flex-wrap:wrap!important}.gradio-container .row>.column,.gradio-container .row>.block,.gradio-container .row>.form{width:100%!important;min-width:0!important;flex-basis:auto!important;box-sizing:border-box!important}.gradio-container .column{min-width:0!important}.ow-section{flex-wrap:wrap}.ow-section h2{font-size:17px}#navigation .tab-container{gap:3px}#navigation .tab-container button{width:100%;text-align:left;font-size:12px!important;padding:10px!important}.ow-eyebrow{letter-spacing:1px}.ow-header{max-width:none!important}.ow-mini-grid{grid-template-columns:1fr 1fr}.ow-mini-grid>div:last-child{grid-column:1/-1}}\n\n#navigation .tab-wrapper,#navigation .tab-container{height:auto!important;overflow:visible!important;max-height:none!important;min-height:44px!important}#navigation .tab-container:after{display:none!important}\n"
# Keep Gradio table surfaces and text consistent even when the visitor uses dark mode.
TABLE_THEME = {}
for _key, _value in {
    'background_fill_primary':'#ffffff', 'background_fill_secondary':'#f4f8f8',
    'body_text_color_subdued':'#54717c', 'border_color_primary':'#cfdee0',
    'table_even_background_fill':'#ffffff', 'table_odd_background_fill':'#f1f6f6',
    'table_border_color':'#cbdadc', 'table_text_color':'#193b48',
    'table_row_focus':'#e3f2ee', 'panel_background_fill':'#ffffff',
    'panel_border_color':'#d7e4e5', 'block_label_text_color':'#54717c',
    'block_title_text_color':'#193b48', 'block_info_text_color':'#54717c',
    'button_secondary_background_fill':'#edf5f3', 'button_secondary_text_color':'#193b48',
}.items():
    TABLE_THEME[_key] = _value
    TABLE_THEME[_key + '_dark'] = _value
CSS += """
.gradio-container table {color:#193b48!important;border-color:#cbdadc!important}
.gradio-container table thead,.gradio-container table th {background:#eaf2f3!important;color:#193b48!important}
.gradio-container table tbody tr:nth-child(odd) {background:#fff!important}
.gradio-container table tbody tr:nth-child(even) {background:#f1f6f6!important}
.gradio-container table td,.gradio-container table th {border-color:#cbdadc!important}
.gradio-container table button,.gradio-container table span {color:#193b48!important}
"""
THEME = gr.themes.Soft(primary_hue='teal',neutral_hue='slate',font=['Noto Sans Thai','Tahoma','Arial','sans-serif']).set(
    **TABLE_THEME,
    body_background_fill='#f4f7f7',body_background_fill_dark='#f4f7f7',
    body_text_color='#193b48',body_text_color_dark='#193b48',
    block_background_fill='white',block_background_fill_dark='white',
    block_border_color='#d7e4e5',block_border_color_dark='#d7e4e5',
    input_background_fill='#f9fbfb',input_background_fill_dark='#f9fbfb',
    input_border_color='#cfdee0',input_border_color_dark='#cfdee0',
    button_primary_background_fill='#16877e',button_primary_background_fill_dark='#16877e',
    button_primary_text_color='white',button_primary_text_color_dark='white',
)

if 'demo' in globals():
    demo.close()
home0 = home_view(DEFAULT)
forecast0 = forecast_view(DEFAULT,MODEL_NAMES[0],SCENARIOS[0])
context0 = task_context(DEFAULT,ROLES[0])
sim0 = simulator_view(DEFAULT,MODEL_NAMES[0],*[float(LATEST.loc[DEFAULT,k]) for k in EDIT])

GRADIO_MAJOR = int(gr.__version__.split('.')[0])
STYLE_ARGS = dict(theme=THEME,css=CSS)
with gr.Blocks(title='OneWaterAI | Ko Sichang',analytics_enabled=False,**(STYLE_ARGS if GRADIO_MAJOR < 6 else {})) as demo:
    gr.HTML('''<header class="ow-header"><div><div class="ow-brand">OneWater<span>AI</span></div><small>อนามัยสิ่งแวดล้อม · อนุรักษ์ทรัพยากรน้ำ</small></div><span class="ow-badge">KO SICHANG / เกาะสีชัง</span></header>
    <div class="ow-eyebrow">MULTI-MARKER × ENVIRONMENT × MACHINE LEARNING</div><h1 class="ow-title">ระบบติดตามแหล่งกำเนิดมลพิษทางน้ำบนเกาะสีชังด้วย AI</h1><p class="ow-subtitle">สำรวจพื้นที่ · ทดสอบสถานการณ์ · ติดตามงานร่วมกับชุมชน</p>
    <div class="ow-banner"><b>ข้อมูลจำลองเพื่อการสาธิตเท่านั้น / SYNTHETIC MODEL EXPERIMENT</b><br>ตัวแปรและฉลากสร้างขึ้นเพื่อทดสอบระบบ ไม่ใช่ผลตรวจหรือความแม่นยำภาคสนามของเกาะสีชัง ไม่ยืนยันความปลอดภัยของน้ำหรืออาหารทะเล · ลิงก์สาธารณะชั่วคราว: ไม่ใส่ข้อมูลส่วนบุคคล · ข้อมูลและรูปเก็บชั่วคราวใน Colab</div>''')
    with gr.Tabs(elem_id='navigation'):
        with gr.Tab('01  Dashboard · ภาพรวม',id='home'):
            with gr.Row():
                gr.HTML(kpi('ลุ่มรับน้ำ / จุดระบาย',12,'B01–B12 · จุดระบายจาก GIS'))
                gr.HTML(kpi('คะแนนเฉลี่ยล่าสุด',f'{LATEST.risk_score.mean():.1f}','DEMO SCORE / 100'))
                gr.HTML(kpi('จุดระดับสูง · สาธิต',int((LATEST.risk_score>=65).sum()),'เกณฑ์สาธิต ≥65'))
                gr.HTML(kpi('ข้อมูลจำลองล่าสุด',f'{LATEST.date.max():%d %b %Y}','450 DAYS · 12 SITES'))
            with gr.Row(equal_height=False):
                with gr.Column(scale=7,min_width=320,elem_classes=['ow-panel']):
                    gr.HTML(section('แผนที่แหล่งรองรับน้ำผิวดินก่อนลงสู่ทะเลเกาะสีชัง','เลือกดูรายละเอียดเมื่อชี้จุด','KO SICHANG'))
                    gis_state = gr.State(INITIAL_GIS)
                    map_component = gr.Plot(map_plot(INITIAL_GIS),show_label=False)
                    layer_choice = gr.CheckboxGroup(LAYER_NAMES,value=LAYER_NAMES,label='Map layers')
                    with gr.Row():
                        raster_choice = gr.Dropdown(RASTER_NAMES,value='None',label='Raster overlay')
                        risk_join = gr.Checkbox(False,label='Illustrative B-ID join: synthetic risk on catchments')
                    gis_note = gr.HTML(gis_status(INITIAL_GIS))
                with gr.Column(scale=4,min_width=300,elem_classes=['ow-panel']):
                    gr.HTML(section('Site overview','ค่าทั้งหมดเป็นข้อมูลจำลอง','LATEST'))
                    hsite = gr.Dropdown(IDS,value=DEFAULT,label='เลือกจุดติดตาม')
                    detail = gr.Markdown(home0[0])
                    env_table = gr.Dataframe(home0[5],interactive=False,label='Environmental · ค่าจำลอง')
                    gr.Markdown('แผนที่ใช้ลุ่มรับน้ำและจุดระบาย B01–B12 จาก R/D8 เดิม · ยังไม่ใช่สถานีตรวจที่ยืนยันภาคสนาม\n\nเลือกจุดจากรายการนี้ได้แม้พื้นแผนที่โหลดไม่สำเร็จ')
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('Historical signals','ประวัติรายวันของจุดที่เลือก · ไม่ใช่ข้อมูลภาคสนาม','450 DAYS'))
                historical = gr.Plot(home0[6],show_label=False)
                tables=[]
                for i,(title,cols) in enumerate(GROUPS):
                    with gr.Accordion(title,open=False):
                        tables.append(gr.Dataframe(home0[i+1],interactive=False))
            hsite.change(home_view,hsite,[detail,*tables,env_table,historical],api_name='site_overview')
            with gr.Accordion('Optional GIS · อัปโหลดชั้นข้อมูลจาก R / GIS',open=False):
                gr.Markdown('''**Original B01–B12 catchments, runoff paths and modeled outlets are already embedded**, regenerated from the original 30 September R analysis and its original DEM/OSM inputs. No geometry was digitized from the screenshot.

Upload any subset below and press **Load GIS files**. Each click overrides only the supplied layers for this browser session; embedded layers remain for other types. To preload for everyone, place files with the documented names in `/content/gis/` before running the application cell. Reload the app after adding preloaded files.

GeoJSON must be **EPSG:4326, longitude–latitude**, near Ko Sichang. Catchments: exactly one Polygon/MultiPolygon per `site_id` B01–B12. Flow paths: LineString/MultiLineString. Optional monitoring points: Point with `site_id` (a subset is allowed). Catchment labels do not establish a verified station association. Risk coloring is an explicit illustrative B-ID join.

Slope: single-band georeferenced GeoTIFF in **degrees (0–90)**. Groundwater screening: single-band GeoTIFF with **1=&lt;5°, 2=5–15°, 3=&gt;15°**, other cells NoData. Defined CRS and NoData required. CRS reprojection is automatic for rasters. GeoJSON limit 20 MB / 300,000 vertices; GeoTIFF limit 100 MB. Rasters display at up to 600 pixels and are visualization layers only.

**Groundwater is an unvalidated slope-only screening proxy.** No measured groundwater availability, aquifer extent, depth, well yield or recharge is inferred. Modeled drainage pathways are not surveyed rivers. User-supplied layers still require provenance and accuracy review.''')
                with gr.Row():
                    catch_file=gr.File(label='catchments_B01_B12.geojson',file_types=['.geojson','.json'],type='filepath')
                    flow_file=gr.File(label='flow_paths.geojson',file_types=['.geojson','.json'],type='filepath')
                    station_file=gr.File(label='monitoring_sites.geojson (optional)',file_types=['.geojson','.json'],type='filepath')
                with gr.Row():
                    slope_file=gr.File(label='terrain_slope.tif (degrees)',file_types=['.tif','.tiff'],type='filepath')
                    groundwater_file=gr.File(label='groundwater_screening.tif (classes 1–3)',file_types=['.tif','.tiff'],type='filepath')
                load_gis_button=gr.Button('Load GIS files · โหลดชั้นข้อมูล',variant='primary')
                load_gis_button.click(upload_gis,[catch_file,flow_file,station_file,slope_file,groundwater_file,layer_choice,raster_choice,risk_join],[gis_state,map_component,gis_note],api_name=False)
            for control in [layer_choice,raster_choice,risk_join]:
                control.change(refresh_map,[gis_state,layer_choice,raster_choice,risk_join],[map_component,gis_note],api_name=False)
            with gr.Accordion('Scientific method & limitations · สูตรและข้อจำกัด',open=False):
                gr.Markdown('\n**Input:** ชื่อ indicator เป็นคนละกลุ่มและมีหน้าที่ต่างกัน ค่าชีวภาพทั้งหมดเป็นดัชนีจำลอง 0–100 ไม่ใช่ copies/L, read counts, ผลบวก หรือความสามารถก่อโรค\n\n**สูตรอ้างอิงสาธิต:**\n\n`score = clip(100 × [0.03 + 0.20H + 0.08A + 0.26PH + 0.15PA + 0.16D + 0.04|pH−8|/1.5 + 0.02max(T−30,0)/5 + R] + noise, 0, 100)`\n\n- H/A = ค่าเฉลี่ยกลุ่ม human/animal หาร 100\n- PH/PA = ค่าเฉลี่ยเป้าหมายเชื้อมนุษย์/สัตว์น้ำหาร 100\n- D = clip((6−DO)/5, 0, 1)\n- R = 0.08×first_rain×tanh(rain/30) + 0.10×tanh(antecedent7/60)×(1−DO/10) − 0.10×tanh(rain/40)×DO/10\n- noise ~ N(0, 2.2)\n\nน้ำหนักและเกณฑ์ทั้งหมดแต่งขึ้น ไม่ใช่มาตรฐานสุขภาพ ฝนมีทั้งพจน์บวกและลบตามบริบท ไม่บังคับว่าฝนต้องเพิ่มคะแนนเสมอ Biodiversity/eDNA ไม่ใช่เชื้อก่อโรค และไม่มีน้ำหนักเพิ่มความเสี่ยงโดยตรงในสูตรนี้\n\n**โมเดล:** Ridge และ MLPRegressor ใช้ predictors เดียวกัน รวม site ID และ rain lag; scaler fit เฉพาะ train; แยก train/test ตามวันที่ของทุกจุด ใช้โมเดลที่ฝึกชุดเดียวกันกับการคาดการณ์และ simulator\n\n**ระดับสาธิต:** ต่ำ <35, ปานกลาง 35–<65, สูง ≥65 ไม่มีระดับใดยืนยันความปลอดภัย\n\n**ข้อจำกัด:** ชื่อ C16S/qC160F เป็น candidate assay labels ตามข้อเสนอ ต้องตรวจคำจำกัดความและความจำเพาะจากวรรณกรรม; MST ไม่ยืนยันเชื้อก่อโรค; eDNA ไม่ยืนยันจำนวนสัตว์หรือความปลอดภัย; เป้าหมายเชื้อไม่ยืนยันการติดเชื้อจริง ผลโมเดลวัดการเลียนแบบสูตรจำลอง ไม่ใช่ความแม่นยำภาคสนาม ไม่ใช้เลือกวัคซีน ยา หรือระยะจับปลา/หมึกที่ปลอดภัย\n\nนำข้อมูลจริงมาแทนได้ด้วย schema เดียวกัน แต่ต้องตรวจหน่วย วิธีตรวจ วันขาดหาย และ outcome ที่เหมาะสมก่อนฝึกใหม่ ไม่ควรคงฉลากจากสูตรจำลองแล้วเรียกว่า validation ภาคสนาม\n')
            gr.File(str(OUT/'synthetic_daily.csv'),label='ดาวน์โหลดข้อมูลจำลอง / schema CSV',interactive=False)
        with gr.Tab('02  Forecast & Action · คาดการณ์และลงมือ',id='forecast'):
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('คาดการณ์สถานการณ์ความเสี่ยงมลพิษล่วงหน้า 30 วัน','ค่าทำนายแบบมีเงื่อนไขตาม inputs สมมติ · ไม่ใช่พยากรณ์อากาศ','CONDITIONAL SCENARIO'))
                with gr.Row():
                    fsite=gr.Dropdown(IDS,value=DEFAULT,label='พื้นที่สำหรับการคาดการณ์และภารกิจ')
                    fmodel=gr.Dropdown(MODEL_NAMES,value=MODEL_NAMES[0],label='โมเดล')
                    scenario=gr.Dropdown(SCENARIOS,value=SCENARIOS[0],label='สมมติฐานอนาคต')
                forecast_button=gr.Button('สร้างสถานการณ์ 30 วัน',variant='primary')
                forecast_plot=gr.Plot(forecast0[0],show_label=False)
                forecast_note=gr.Markdown(forecast0[2])
                with gr.Accordion('รายละเอียดรายวัน / ฝนหน่วงเวลา / first-rain',open=False):
                    forecast_table=gr.Dataframe(forecast0[1],interactive=False)
                f_inputs=[fsite,fmodel,scenario]; f_outputs=[forecast_plot,forecast_table,forecast_note]
                forecast_button.click(forecast_view,f_inputs,f_outputs,api_name='forecast')
                for c in f_inputs:
                    c.change(forecast_view,f_inputs,f_outputs,api_name=False)
            with gr.Row(equal_height=False):
                with gr.Column(scale=4,min_width=300,elem_classes=['ow-panel']):
                    gr.HTML(section('Model evaluation','เปรียบเทียบ predictors และช่วงเวลาชุดเดียวกัน','TIME SPLIT'))
                    gr.Plot(performance_plot(),show_label=False)
                with gr.Column(scale=6,min_width=320,elem_classes=['ow-panel']):
                    gr.Markdown(f'### ผลทดสอบคะแนนรายวันตามลำดับเวลา\nTrain {TRAIN.date.min():%d/%m/%Y}–{TRAIN.date.max():%d/%m/%Y}; Test {TEST.date.min():%d/%m/%Y}–{TEST.date.max():%d/%m/%Y}\n\nผลรวม 12 จุด ใช้ input ของวันทดสอบ ไม่ใช่ backtest ความแม่นยำพยากรณ์ 30 วัน หรือ validation ภาคสนาม')
                    gr.Dataframe(METRICS.round(3),interactive=False)
            with gr.Column(elem_classes=['ow-panel']):
                gr.HTML(section('ข้อเสนอแนะกิจกรรมชุมชนเพื่อบรรเทาความเสี่ยง','ข้อเสนอแนะตามบทบาท · อ้างอิงคะแนนปัจจุบันจำลอง','7 ROLES / 4 TASKS'))
                role=gr.Dropdown(ROLES,value=ROLES[0],label='เลือกบทบาทของคุณ')
                task_state=gr.State(context0)
                task_md=gr.Markdown(task_summary(context0))
                heads,responses,images=[],[],[]
                for start in [0,2]:
                    with gr.Row():
                        for i in range(start,start+2):
                            with gr.Column(min_width=280,elem_classes=['ow-task']):
                                heads.append(gr.Markdown(task_headers(context0)[i]))
                                responses.append(gr.Textbox(label=f'คำตอบภารกิจ {i+1}',lines=3,max_lines=6))
                                images.append(gr.Image(label=f'รูปภารกิจ {i+1} (ไม่บังคับ)',sources=['upload'],type='filepath',format='png',height=150))
                gr.Markdown(ACTION_NOTE)
                submit=gr.Button('ส่งคำตอบทั้ง 4 ภารกิจ',variant='primary')
                confirmation=gr.Markdown()
                receipt=gr.File(label='ใบรับการส่งครั้งนี้ (CSV)',interactive=False)
                reset_outputs=[task_state,task_md,*heads,*responses,*images,confirmation,receipt]
                fsite.change(reset_tasks,[fsite,role],reset_outputs,api_name=False)
                role.change(reset_tasks,[fsite,role],reset_outputs,api_name=False)
                submit.click(save_tasks,[task_state,fsite,role,*responses,*images],[confirmation,receipt],api_name=False)
                gr.Markdown('คำตอบและรูปเก็บชั่วคราวใน runtime ภายใต้ onewaterai_outputs; ไม่ใส่ข้อมูลส่วนบุคคล ดาวน์โหลดใบรับก่อนปิด Colab การส่งงานไม่เปลี่ยนสถานะคุณภาพน้ำ')
        with gr.Tab('03  Risk Simulator · ทดลองประเมิน',id='simulator'):
            gr.HTML('''<div class="ow-flow"><div><small>01 · SOURCE</small><strong>เกาะสีชัง B01–B12</strong></div><div><small>02 · DATA</small><strong>สิ่งแวดล้อม + MST + eDNA</strong></div><div><small>03 · MODEL</small><strong>Ridge / MLP 32 → 16 → 1</strong></div><div><small>04 · RISK SCORE</small><strong>คะแนนสาธิต 0–100</strong></div><div><small>05 · ACTION</small><strong>งานตามกฎที่อธิบายได้</strong></div></div>''')
            with gr.Row(equal_height=False):
                with gr.Column(scale=6,min_width=320,elem_classes=['ow-panel']):
                    gr.HTML(section('1. เลือกจุดและปรับข้อมูลนำเข้าจริง','ตัวชี้วัดชีวภาพทุกตัวเป็นดัชนีจำลอง 0–100','EDITABLE INPUTS'))
                    with gr.Row():
                        ssite=gr.Dropdown(IDS,value=DEFAULT,label='จุดติดตาม')
                        smodel=gr.Dropdown(MODEL_NAMES,value=MODEL_NAMES[0],label='โมเดลที่ฝึกแล้ว')
                    gr.HTML('<div class="ow-action"><strong>Ko Sichang / เกาะสีชัง</strong>B01–B12 เป็นลุ่มรับน้ำและจุดระบายจาก GIS เดิม · ไม่ใช่สถานีตรวจยืนยัน<br>ค่าคุณภาพน้ำและคะแนนเป็นข้อมูลจำลอง</div>')
                    gr.Markdown('### ENVIRONMENTAL · ค่าจำลอง')
                    edits=[]
                    for start in [0,2]:
                        with gr.Row():
                            for name,label in zip(ENV[start:start+2],ENV_LABELS[start:start+2]):
                                lo,hi=RANGES[name]
                                edits.append(gr.Slider(lo,hi,value=float(LATEST.loc[DEFAULT,name]),step=.01,label=label))
                    for title,cols in GROUPS:
                        with gr.Accordion(title,open=False):
                            for start in range(0,len(cols),2):
                                with gr.Row():
                                    for name in cols[start:start+2]:
                                        edits.append(gr.Number(float(LATEST.loc[DEFAULT,name]),label=name,minimum=0,maximum=100,precision=2))
                    lag_md=gr.Markdown(load_sim_site(DEFAULT)[-1])
                    assess_button=gr.Button('ประเมินความเสี่ยง · Run assessment',variant='primary')
                    reset_button=gr.Button('คืนค่าล่าสุดของจุดที่เลือก · Reset inputs')
                with gr.Column(scale=5,min_width=320,elem_classes=['ow-panel']):
                    evaluation_state=gr.State(input_fingerprint(DEFAULT,MODEL_NAMES[0],[float(LATEST.loc[DEFAULT,k]) for k in EDIT]))
                    result=gr.HTML(sim0[0])
                    explanation=gr.Plot(sim0[1],show_label=False)
                    sim_note=gr.Markdown(sim0[2])
                    gr.Markdown('Sensitivity คือผลต่างจากการแทนทีละตัวด้วยค่าเฉลี่ย train ไม่ใช่ causal effect หรือ feature importance ทั่วไป')
            sim_outputs=[result,explanation,sim_note,evaluation_state]
            assess_button.click(simulator_assess,[ssite,smodel,*edits],sim_outputs,api_name='simulate')
            ssite.change(reset_simulator,[ssite,smodel],[*edits,lag_md,*sim_outputs],api_name=False)
            reset_button.click(reset_simulator,[ssite,smodel],[*edits,lag_md,*sim_outputs],api_name=False)
            for c in [smodel,*edits]:
                c.input(pending_result,[evaluation_state,ssite,smodel,*edits],result,queue=False,api_name=False)
            gr.Markdown('เปลี่ยนโมเดลหรือ input แล้วกดประเมิน · เปลี่ยนจุดจะคืนค่า input และคำนวณใหม่ · ผลประเมินไม่ใช่คำยืนยันความปลอดภัย')
    gr.HTML('<footer class="ow-footer"><span>OneWaterAI · Ko Sichang environmental health prototype</span><span>SYNTHETIC DATA · UNVALIDATED MODELS · TEMPORARY COLAB STORAGE</span></footer>')

demo.queue(default_concurrency_limit=2,max_size=24)


def launch_app(share=True, **kwargs):
    """Colab: public temporary URL, light mode. Local tests use share=False."""
    app, local_url, share_url = demo.launch(share=share,show_error=True,ssr_mode=False,
        max_file_size='100mb',inbrowser=False,**(STYLE_ARGS if GRADIO_MAJOR >= 6 else {}),**kwargs)
    if share_url:
        print('Share this temporary OneWaterAI URL:',share_url+'?__theme=light')
        print('Keep the Colab runtime running. Download receipts before ending the session.')
    return app, local_url, share_url


if __name__ == '__main__' and os.environ.get('ONEWATERAI_NO_LAUNCH') != '1':
    launch_app(share=True, debug=True)


Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://83de16e5fa1c612ff2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
